In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__notebook__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__output__.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__results__.html
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/custom.css
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_latest.pth
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/config/environment.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/config/training_summary.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/data_split_indices.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/data/dataset_manifest.json
/kaggle/input/notebooks/crah

In [3]:

# =============================================================================
# JFAMP — PRE-EXPERIMENT AUDIT
# STEP 0A: INVENTORY PREVIOUS NOTEBOOK INPUTS AND PROJECT ARTIFACTS
# =============================================================================

import os
import json
from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

print("=" * 85)
print("JFAMP — PRE-EXPERIMENT ARTIFACT AUDIT")
print("=" * 85)

# -------------------------------------------------------------------------
# 1. List mounted Kaggle inputs
# -------------------------------------------------------------------------

print("\n[1] MOUNTED INPUT DIRECTORIES")
print("-" * 85)

input_dirs = sorted([
    p for p in INPUT_ROOT.iterdir()
    if p.is_dir()
])

for i, directory in enumerate(input_dirs, 1):
    print(f"{i}. {directory}")

# -------------------------------------------------------------------------
# 2. Recursively inventory files
# -------------------------------------------------------------------------

print("\n[2] FILE INVENTORY")
print("-" * 85)

all_files = []

for directory in input_dirs:
    for path in directory.rglob("*"):
        if path.is_file():
            try:
                size = path.stat().st_size
            except OSError:
                size = -1

            all_files.append({
                "input": directory.name,
                "filename": path.name,
                "extension": path.suffix.lower(),
                "size_mb": round(size / (1024 ** 2), 3),
                "path": str(path)
            })

inventory = pd.DataFrame(all_files)

if len(inventory):
    inventory = inventory.sort_values(
        ["input", "filename"]
    ).reset_index(drop=True)

    print("Total mounted files:", len(inventory))
    display(inventory)
else:
    print("No files found under /kaggle/input.")

# -------------------------------------------------------------------------
# 3. Search for important JFAMP artifacts
# -------------------------------------------------------------------------

print("\n[3] IMPORTANT JFAMP ARTIFACT SEARCH")
print("-" * 85)

keywords = [
    "mobilenetv2_cifar100_best",
    "stage5_unified_characterization",
    "stage5c6_recovered_allocation",
    "stage5c6_verified_candidate",
    "stage5c6_verified_predictions",
    "final_mixed_precision_model",
    "final_layerwise_allocation",
    "final_validation_predictions",
    "stage6a_protection_raw_trials",
    "stage6a_protection_summary",
    "stage6a_layer_protection_summary",
    "stage6a_protection_storage",
    "stage6c_joint_allocation",
    "jfamp_mobilenetv2_joint_final",
    "jfamp_final_layerwise_allocation",
    "jfamp_final_research_report"
]

matched = []

for item in all_files:
    filename_lower = item["filename"].lower()

    for keyword in keywords:
        if keyword.lower() in filename_lower:
            matched.append({
                "matched_keyword": keyword,
                "filename": item["filename"],
                "size_mb": item["size_mb"],
                "path": item["path"]
            })

matched_df = pd.DataFrame(matched)

if len(matched_df):
    display(matched_df)
else:
    print("No matching artifacts found.")

# -------------------------------------------------------------------------
# 4. Identify notebook files
# -------------------------------------------------------------------------

print("\n[4] PREVIOUS NOTEBOOK FILES")
print("-" * 85)

notebooks = [
    item for item in all_files
    if item["extension"] == ".ipynb"
]

for notebook in notebooks:
    print(notebook["path"])

# -------------------------------------------------------------------------
# 5. Inspect checkpoint candidates without loading them
# -------------------------------------------------------------------------

print("\n[5] CHECKPOINT CANDIDATES")
print("-" * 85)

checkpoints = [
    item for item in all_files
    if item["extension"] in [".pth", ".pt", ".ckpt"]
]

for checkpoint in checkpoints:
    print(
        f"{checkpoint['filename']:<55} "
        f"{checkpoint['size_mb']:>10.3f} MB"
    )
    print("  ", checkpoint["path"])

# -------------------------------------------------------------------------
# 6. Inspect CSV schemas for key experiment results
# -------------------------------------------------------------------------

print("\n[6] IMPORTANT CSV SCHEMAS")
print("-" * 85)

schema_keywords = [
    "stage5_unified_characterization",
    "stage5c6_recovered_allocation",
    "final_layerwise_allocation",
    "stage6a_protection_summary",
    "stage6a_protection_storage",
    "stage6c_joint_allocation"
]

for keyword in schema_keywords:

    candidates = [
        item for item in all_files
        if keyword in item["filename"].lower()
    ]

    for item in candidates:
        try:
            frame = pd.read_csv(item["path"], nrows=3)

            print(f"\n{item['filename']}")
            print("Shape (first 3 rows):", frame.shape)
            print("Columns:", list(frame.columns))

        except Exception as exc:
            print(f"Could not inspect {item['filename']}: {exc}")

# -------------------------------------------------------------------------
# 7. Inspect JSON experiment summaries
# -------------------------------------------------------------------------

print("\n[7] IMPORTANT JSON REPORTS")
print("-" * 85)

json_keywords = [
    "stage5d",
    "stage6a_protection_report",
    "stage6c_optimization_summary",
    "jfamp_final_research_report"
]

for item in all_files:

    if item["extension"] != ".json":
        continue

    if not any(k in item["filename"].lower() for k in json_keywords):
        continue

    print(f"\nREPORT: {item['filename']}")

    try:
        with open(item["path"], "r") as f:
            report = json.load(f)

        print(json.dumps(report, indent=2)[:5000])

    except Exception as exc:
        print("Could not read report:", exc)

# -------------------------------------------------------------------------
# 8. Save an audit inventory in working storage
# -------------------------------------------------------------------------

audit_path = WORK_ROOT / "jfamp_preexperiment_inventory.csv"

if len(inventory):
    inventory.to_csv(audit_path, index=False)
    print("\nInventory saved:", audit_path)

print("\n" + "=" * 85)
print("PRE-CHECK 0A COMPLETE")
print("No model weights, checkpoints or experiment results were modified.")
print("=" * 85)


JFAMP — PRE-EXPERIMENT ARTIFACT AUDIT

[1] MOUNTED INPUT DIRECTORIES
-------------------------------------------------------------------------------------
1. /kaggle/input/notebooks

[2] FILE INVENTORY
-------------------------------------------------------------------------------------
Total mounted files: 129


,input,filename,extension,size_mb,path
0,notebooks,BF16_1bit_corrupted_payload.bin,.bin,0.244,/kaggle/input/notebooks/crahulnaidu/mobile-net...
1,notebooks,BF16_8bit_corrupted_payload.bin,.bin,0.244,/kaggle/input/notebooks/crahulnaidu/mobile-net...
2,notebooks,E4M11_SN_1bit_corrupted_payload.bin,.bin,0.244,/kaggle/input/notebooks/crahulnaidu/mobile-net...
3,notebooks,E4M11_SN_8bit_corrupted_payload.bin,.bin,0.244,/kaggle/input/notebooks/crahulnaidu/mobile-net...
4,notebooks,FP16_1bit_corrupted_payload.bin,.bin,0.244,/kaggle/input/notebooks/crahulnaidu/mobile-net...
...,...,...,...,...,...
124,notebooks,stage6c_optimization_summary.json,.json,0.001,/kaggle/input/notebooks/crahulnaidu/mobile-net...
125,notebooks,test,,29.611,/kaggle/input/notebooks/crahulnaidu/mobile-net...
126,notebooks,train,,148.058,/kaggle/input/notebooks/crahulnaidu/mobile-net...
127,notebooks,training_history.csv,.csv,0.016,/kaggle/input/notebooks/crahulnaidu/mobile-net...



[3] IMPORTANT JFAMP ARTIFACT SEARCH
-------------------------------------------------------------------------------------


,matched_keyword,filename,size_mb,path
0,mobilenetv2_cifar100_best,mobilenetv2_cifar100_best.pth,9.211,/kaggle/input/notebooks/crahulnaidu/mobile-net...
1,stage5_unified_characterization,stage5_unified_characterization.csv,0.163,/kaggle/input/notebooks/crahulnaidu/mobile-net...
2,stage5c6_recovered_allocation,stage5c6_recovered_allocation.csv,0.005,/kaggle/input/notebooks/crahulnaidu/mobile-net...
3,stage5c6_verified_candidate,stage5c6_verified_candidate.pth,9.215,/kaggle/input/notebooks/crahulnaidu/mobile-net...
4,stage5c6_verified_predictions,stage5c6_verified_predictions.npz,0.022,/kaggle/input/notebooks/crahulnaidu/mobile-net...
5,stage6a_layer_protection_summary,stage6a_layer_protection_summary.csv,0.177,/kaggle/input/notebooks/crahulnaidu/mobile-net...
6,stage6a_protection_raw_trials,stage6a_protection_raw_trials.csv,21.008,/kaggle/input/notebooks/crahulnaidu/mobile-net...
7,stage6a_protection_storage,stage6a_protection_storage.csv,0.088,/kaggle/input/notebooks/crahulnaidu/mobile-net...
8,stage6a_protection_summary,stage6a_protection_summary.csv,0.002,/kaggle/input/notebooks/crahulnaidu/mobile-net...
9,stage6c_joint_allocation,stage6c_joint_allocation.csv,0.004,/kaggle/input/notebooks/crahulnaidu/mobile-net...



[4] PREVIOUS NOTEBOOK FILES
-------------------------------------------------------------------------------------
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/__notebook__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/__notebook__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/__notebook__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/.virtual_documents/__notebook_source__.ipynb
/kaggle/input/notebooks/crahulnaidu/mobile-net-100/.virtual_documents/__notebook_source__.ipynb

[5] CHECKPOINT CANDIDATES
-------------------------------------------------------------------------------------
mobilenetv2_cifar100_best.pth                                9.211 MB
   /kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
stage5c3_mixed_precision_model.pth                           9.216 MB
   /kaggle

In [4]:

# =============================================================================
# JFAMP — PRE-EXPERIMENT AUDIT
# STEP 0B: CHECKPOINT, BASELINE, SPLIT AND RESULT CONSISTENCY
# =============================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd
import torch

from pathlib import Path

ROOT = Path("/kaggle/input/notebooks/crahulnaidu")

N1 = ROOT / "mobile-net-100"
N2 = ROOT / "mobile-net-100-2"
N3 = ROOT / "mobile-net-100-3"

PROJECT1 = N1 / "JFAMP_MobileNetV2"
PROJECT2 = N2 / "JFAMP_MobileNetV2"
PROJECT3 = N3 / "JFAMP_MobileNetV2"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def inspect_npz(path):
    z = np.load(path, allow_pickle=False)
    result = {}
    for key in z.files:
        arr = z[key]
        result[key] = {
            "shape": arr.shape,
            "dtype": str(arr.dtype)
        }
    return result

print("=" * 85)
print("JFAMP — PRE-EXPERIMENT VERIFICATION")
print("=" * 85)

# -------------------------------------------------------------------------
# 1. Original checkpoint integrity
# -------------------------------------------------------------------------

print("\n[1] ORIGINAL CHECKPOINT INTEGRITY")
print("-" * 85)

checkpoint_paths = {
    "mobile-net-100": (
        PROJECT1 / "checkpoints/mobilenetv2_cifar100_best.pth"
    ),
    "mobile-net-100-2": (
        PROJECT2 / "checkpoints/mobilenetv2_cifar100_best.pth"
    ),
    "mobile-net-100-3": (
        PROJECT3 / "checkpoints/mobilenetv2_cifar100_best.pth"
    )
}

checkpoint_hashes = {}

for name, path in checkpoint_paths.items():
    assert path.exists(), f"Missing checkpoint: {path}"

    checkpoint_hashes[name] = sha256(path)

    print(f"\n{name}")
    print("Path:", path)
    print("Size:", round(path.stat().st_size / 1024**2, 3), "MB")
    print("SHA256:", checkpoint_hashes[name])

unique_hashes = set(checkpoint_hashes.values())

print("\nCheckpoint copies identical:", len(unique_hashes) == 1)

# Inspect original checkpoint structure.
checkpoint_path = checkpoint_paths["mobile-net-100-3"]

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location="cpu")

print("\nCheckpoint object type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys())[:30])

    state = checkpoint.get(
        "state_dict",
        checkpoint.get("model_state_dict", checkpoint)
    )

    if isinstance(state, dict):
        tensors = [
            (k, v) for k, v in state.items()
            if isinstance(v, torch.Tensor)
        ]

        print("Tensor count:", len(tensors))
        print(
            "Total tensor parameters:",
            sum(v.numel() for _, v in tensors)
        )
        print("First 10 tensor keys:")
        for k, v in tensors[:10]:
            print(k, tuple(v.shape), v.dtype)

del checkpoint

# -------------------------------------------------------------------------
# 2. Dataset split integrity
# -------------------------------------------------------------------------

print("\n[2] DATASET SPLIT")
print("-" * 85)

split_path = PROJECT1 / "data/data_split_indices.json"
manifest_path = PROJECT1 / "data/dataset_manifest.json"

assert split_path.exists(), f"Missing split file: {split_path}"
assert manifest_path.exists(), f"Missing dataset manifest: {manifest_path}"

split = load_json(split_path)
manifest = load_json(manifest_path)

print("Split file SHA256:", sha256(split_path))
print("Split top-level type:", type(split).__name__)

if isinstance(split, dict):
    print("Split keys:", list(split.keys()))

    for key, value in split.items():
        if isinstance(value, list):
            print(key, "count:", len(value))
        elif isinstance(value, dict):
            print(
                key,
                "subkeys:",
                list(value.keys())[:15]
            )

print("\nDataset manifest:")
print(json.dumps(manifest, indent=2)[:4000])

# -------------------------------------------------------------------------
# 3. FP32 baseline reports
# -------------------------------------------------------------------------

print("\n[3] FP32 BASELINE REPORTS")
print("-" * 85)

baseline_files = [
    PROJECT2 / "experiments/stage_02_baseline/fp32_baseline.json",
    PROJECT2 / "experiments/stage_02_baseline/fp32_reference_manifest.json"
]

for path in baseline_files:
    print("\nFILE:", path.name)

    if path.exists():
        report = load_json(path)
        print(json.dumps(report, indent=2)[:5000])
    else:
        print("MISSING")

# -------------------------------------------------------------------------
# 4. FP32 prediction files
# -------------------------------------------------------------------------

print("\n[4] FP32 PREDICTION FILES")
print("-" * 85)

prediction_paths = {
    "results_reference": (
        PROJECT3 / "results/fp32_reference_predictions.npz"
    ),
    "stage2_reference_mobile_net_100_2": (
        PROJECT2 /
        "experiments/stage_02_baseline/fp32_reference_predictions.npz"
    ),
    "stage2_reference_mobile_net_100_3": (
        PROJECT3 /
        "experiments/stage_02_baseline/fp32_reference_predictions.npz"
    )
}

prediction_info = {}

for name, path in prediction_paths.items():
    print("\n", name)

    if not path.exists():
        print("MISSING")
        continue

    print("Size MB:", round(path.stat().st_size / 1024**2, 3))
    print("SHA256:", sha256(path))

    info = inspect_npz(path)
    prediction_info[name] = info

    for key, value in info.items():
        print(key, value)

# -------------------------------------------------------------------------
# 5. Stage 5D final report and saved predictions
# -------------------------------------------------------------------------

print("\n[5] STAGE 5D FINAL VALIDATION")
print("-" * 85)

stage5d = (
    PROJECT3 /
    "experiments/stage_05_allocation/stage5d_final_verified"
)

stage5_report_path = stage5d / "final_research_report.json"
stage5_predictions_path = stage5d / "final_validation_predictions.npz"

if stage5_report_path.exists():
    stage5_report = load_json(stage5_report_path)
    print("Stage 5D report:")
    print(json.dumps(stage5_report, indent=2)[:6000])
else:
    print("Stage 5D report missing.")

if stage5_predictions_path.exists():
    print("\nStage 5D prediction arrays:")
    for k, v in inspect_npz(stage5_predictions_path).items():
        print(k, v)

# -------------------------------------------------------------------------
# 6. Stage 6D final report
# -------------------------------------------------------------------------

print("\n[6] STAGE 6D FINAL REPORT")
print("-" * 85)

stage6_report_path = (
    PROJECT3 /
    "experiments/stage_06_protection/stage6d_final/"
    "jfamp_final_research_report.json"
)

if stage6_report_path.exists():
    stage6_report = load_json(stage6_report_path)
    print(json.dumps(stage6_report, indent=2)[:7000])
else:
    print("Stage 6D report missing.")

# -------------------------------------------------------------------------
# 7. Global quantization results
# -------------------------------------------------------------------------

print("\n[7] GLOBAL QUANTIZATION RESULTS")
print("-" * 85)

global_quant_path = (
    PROJECT2 /
    "experiments/stage_02_baseline/global_quantization_results.csv"
)

if global_quant_path.exists():
    global_quant = pd.read_csv(global_quant_path)
    print("Shape:", global_quant.shape)
    print("Columns:", list(global_quant.columns))
    display(global_quant)
else:
    print("Global quantization results missing.")

# -------------------------------------------------------------------------
# 8. Stage 5 and Stage 6 allocation comparison
# -------------------------------------------------------------------------

print("\n[8] ALLOCATION FILES")
print("-" * 85)

allocation_paths = {
    "stage5_final": (
        stage5d / "final_layerwise_allocation.csv"
    ),
    "stage6_final": (
        PROJECT3 /
        "experiments/stage_06_protection/stage6d_final/"
        "jfamp_final_layerwise_allocation.csv"
    )
}

for name, path in allocation_paths.items():
    print("\n", name)

    if path.exists():
        frame = pd.read_csv(path)
        print("Rows:", len(frame))
        print("Columns:", list(frame.columns))
        display(frame.head(5))
    else:
        print("MISSING")

# -------------------------------------------------------------------------
# 9. Summary
# -------------------------------------------------------------------------

print("\n" + "=" * 85)
print("PRE-EXPERIMENT VERIFICATION SUMMARY")
print("=" * 85)

print("Checkpoint copies identical:", len(unique_hashes) == 1)
print("Dataset split available:", split_path.exists())
print("Dataset manifest available:", manifest_path.exists())
print("Stage 5D report available:", stage5_report_path.exists())
print("Stage 5D predictions available:", stage5_predictions_path.exists())
print("Stage 6D report available:", stage6_report_path.exists())
print("Global quantization results available:", global_quant_path.exists())

print("\nBaseline prediction files found:")
for name in prediction_info:
    print("-", name)

print("\nAUDIT COMPLETE — NO EXISTING ARTIFACTS MODIFIED")
print("=" * 85)


JFAMP — PRE-EXPERIMENT VERIFICATION

[1] ORIGINAL CHECKPOINT INTEGRITY
-------------------------------------------------------------------------------------

mobile-net-100
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
Size: 9.211 MB
SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a

mobile-net-100-2
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
Size: 9.211 MB
SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a

mobile-net-100-3
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth
Size: 9.211 MB
SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a

Checkpoint copies identical: True

Checkpoint object type: <class 'dict'>
Top-level keys: ['epoch', 'model_state_dict', 'best_val_accuracy', 'val_loss', 'seed', 'architect

,format,accuracy_percent,accuracy_delta_pp,validation_loss,sqnr_db,payload_bytes,payload_mib,storage_reduction_percent
0,FP16,68.06,0.00,1.161094,73.600582,4635520,4.420776,50.0
1,BF16,68.08,0.02,1.161554,55.567143,4635520,4.420776,50.0
2,E4M11_SN,68.06,0.00,1.161071,78.346405,4635520,4.420776,50.0
3,FP8_E4M3FN,66.42,-1.64,1.223085,30.879418,2317760,2.210388,75.0
4,FP8_E5M2,65.06,-3.00,1.270926,25.559400,2317760,2.210388,75.0
5,INT8,67.94,-0.12,1.167120,37.454353,2317760,2.210388,75.0
6,INT4,38.38,-29.68,2.828018,12.835035,1158880,1.105194,87.5



[8] ALLOCATION FILES
-------------------------------------------------------------------------------------

 stage5_final
Rows: 53
Columns: ['layer', 'datatype', 'weight_count', 'raw_accuracy_delta_pp', 'payload_bytes', 'quantization_loss_pp', 'datatype_bits', 'storage_bits', 'candidate_source', 'storage_reduction_pct', 'protection', 'protection_validated']


,layer,datatype,weight_count,raw_accuracy_delta_pp,payload_bytes,quantization_loss_pp,datatype_bits,storage_bits,candidate_source,storage_reduction_pct,protection,protection_validated
0,classifier.1,FP8_E4M3FN,128000,0.04,128000.0,0.00,8,1024000,measured_quantization,75.0,none,False
1,features.0.0,E4M11_SN,864,0.00,1728.0,-0.00,16,13824,measured_quantization,50.0,none,False
2,features.1.conv.0.0,BF16,288,0.04,576.0,0.00,16,4608,measured_quantization,50.0,none,False
3,features.1.conv.1,FP8_E5M2,512,0.00,512.0,-0.00,8,4096,measured_quantization,75.0,none,False
4,features.10.conv.0.0,INT4,24576,-0.06,12288.0,0.06,4,98304,measured_quantization,87.5,none,False



 stage6_final
Rows: 53
Columns: ['layer', 'format', 'strategy', 'Q_quantization_loss_pp', 'R_residual_fault_risk_percent', 'R_fraction', 'S_payload_bytes', 'S_protected_bytes', 'overhead_percent', 'objective_cost']


,layer,format,strategy,Q_quantization_loss_pp,R_residual_fault_risk_percent,R_fraction,S_payload_bytes,S_protected_bytes,overhead_percent,objective_cost
0,classifier.1,INT4,secded,0.0,50.0,0.5,64000,78000,21.875,0.5
1,features.0.0,INT4,secded,0.0,50.0,0.5,432,527,21.875,0.5
2,features.1.conv.0.0,INT4,secded,0.0,50.0,0.5,144,176,21.875,0.5
3,features.1.conv.1,INT4,secded,0.0,50.0,0.5,256,312,21.875,0.5
4,features.10.conv.0.0,INT4,secded,0.0,50.0,0.5,12288,14976,21.875,0.5



PRE-EXPERIMENT VERIFICATION SUMMARY
Checkpoint copies identical: True
Dataset split available: True
Dataset manifest available: True
Stage 5D report available: True
Stage 5D predictions available: True
Stage 6D report available: True
Global quantization results available: True

Baseline prediction files found:
- results_reference
- stage2_reference_mobile_net_100_2
- stage2_reference_mobile_net_100_3

AUDIT COMPLETE — NO EXISTING ARTIFACTS MODIFIED


In [5]:
# ============================================================
# PRE-CHECK 0C — RECOVER ORIGINAL EXPERIMENT IMPLEMENTATION
# Read-only inspection of the three source notebooks
# ============================================================

from pathlib import Path
import json
import re

ROOT = Path("/kaggle/input/notebooks/crahulnaidu")

NOTEBOOKS = {
    "original_training": ROOT / "mobile-net-100" / "__notebook__.ipynb",
    "stage2_stage3": ROOT / "mobile-net-100-2" / "__notebook__.ipynb",
    "stage4_stage5_stage6": ROOT / "mobile-net-100-3" / "__notebook__.ipynb",
}

KEYWORDS = [
    "class MobileNetV2",
    "def evaluate",
    "def get_model",
    "transforms.Normalize",
    "validation_indices",
    "encode_weights",
    "decode_weights",
    "inject_bit_flips",
    "def load",
    "CIFAR100",
]

for notebook_name, notebook_path in NOTEBOOKS.items():

    print("\n" + "=" * 100)
    print("NOTEBOOK:", notebook_name)
    print("PATH:", notebook_path)
    print("=" * 100)

    if not notebook_path.exists():
        print("NOTEBOOK FILE NOT FOUND")
        continue

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])
    print("Total cells:", len(cells))

    for cell_idx, cell in enumerate(cells):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        matched = [
            keyword for keyword in KEYWORDS
            if keyword.lower() in source.lower()
        ]

        if not matched:
            continue

        print("\n" + "-" * 90)
        print(f"CELL {cell_idx} | MATCHES: {matched}")
        print("-" * 90)

        # Print source for relevant implementation cells.
        print(source[:18000])

        if len(source) > 18000:
            print("\n... CELL TRUNCATED ...")

print("\n" + "=" * 100)
print("SOURCE INSPECTION COMPLETE — NO FILES MODIFIED")
print("=" * 100)


NOTEBOOK: original_training
PATH: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/__notebook__.ipynb
Total cells: 4

------------------------------------------------------------------------------------------
CELL 1 | MATCHES: ['validation_indices', 'CIFAR100']
------------------------------------------------------------------------------------------

# ============================================================
# JFAMP — MobileNetV2 + CIFAR-100
# STAGE 0, CELL 1: Workspace, Dataset and Reproducibility Setup
# ============================================================

import os
import sys
import json
import time
import random
import hashlib
import platform
from pathlib import Path

import numpy as np
import torch
import torchvision
from torchvision import datasets

# ---------------- CONFIGURATION ----------------

SEED = 42

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
DATA_ROOT = ROOT / "data" / "torchvision"

# ---------------- REPRODUCIBILITY ----------------

random.see

In [6]:
# ============================================================
# JFAMP — PRE-CHECK 0D
# REPRODUCE ORIGINAL FP32 BASELINE
# ============================================================

import json
import hashlib
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from pathlib import Path
from torch.utils.data import DataLoader, Subset
from torchvision.models import mobilenet_v2

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

SOURCE = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100/"
    "JFAMP_MobileNetV2"
)

CHECKPOINT_PATH = SOURCE / "checkpoints/mobilenetv2_cifar100_best.pth"
SPLIT_PATH = SOURCE / "data/data_split_indices.json"
DATA_ROOT = SOURCE / "data/torchvision"

REFERENCE_PATH = (
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/")
    / "JFAMP_MobileNetV2/experiments/stage_02_baseline/"
    / "fp32_reference_predictions.npz"
)

OUTPUT_DIR = (
    Path("/kaggle/working/JFAMP_MobileNetV2/experiments/")
    / "revised_jfamp_fault_aware/precheck_fp32_reproduction"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 256
NUM_WORKERS = 2

MEAN = (0.5071, 0.4867, 0.4408)
STD = (0.2675, 0.2565, 0.2761)

EXPECTED_CHECKPOINT_SHA256 = (
    "07c3c343451dcef593efc9172b9e07194"
    "b72f7c8c635b115f03104d7a8b2ff0a"
)

EXPECTED_SPLIT_SHA256 = (
    "e7eaa993c06cb2f5eb9aa8ce111bb8e78"
    "ad7a87aa369f04a272e773d23a1039b"
)

# ------------------------------------------------------------
# 2. REPRODUCIBILITY SETTINGS
# ------------------------------------------------------------

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 75)
print("JFAMP — ORIGINAL FP32 BASELINE REPRODUCTION")
print("=" * 75)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

# ------------------------------------------------------------
# 3. VERIFY CHECKPOINT AND SPLIT IDENTITIES
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)

    return h.hexdigest()


assert CHECKPOINT_PATH.exists(), "Checkpoint missing."
assert SPLIT_PATH.exists(), "Canonical split missing."
assert REFERENCE_PATH.exists(), "Original FP32 reference missing."

checkpoint_hash = sha256_file(CHECKPOINT_PATH)
split_hash = sha256_file(SPLIT_PATH)

assert checkpoint_hash == EXPECTED_CHECKPOINT_SHA256, (
    "Checkpoint hash mismatch."
)

assert split_hash == EXPECTED_SPLIT_SHA256, (
    "Validation split hash mismatch."
)

print("\nIDENTITY VERIFICATION")
print("Checkpoint SHA256:", checkpoint_hash)
print("Split SHA256:", split_hash)
print("Checkpoint identity: VERIFIED")
print("Split identity: VERIFIED")

# ------------------------------------------------------------
# 4. LOAD CANONICAL VALIDATION INDICES
# ------------------------------------------------------------

with open(SPLIT_PATH, "r") as f:
    split = json.load(f)

val_indices = split["validation_indices"]

assert len(val_indices) == 5000
assert len(set(val_indices)) == 5000

# ------------------------------------------------------------
# 5. RECONSTRUCT ORIGINAL VALIDATION DATASET
# ------------------------------------------------------------

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_dataset = torchvision.datasets.CIFAR100(
    root=str(DATA_ROOT),
    train=True,
    download=False,
    transform=eval_transform,
)

assert len(full_dataset) == 50000

val_dataset = Subset(full_dataset, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    drop_last=False,
)

print("\nDATASET")
print("Training archive:", len(full_dataset))
print("Validation samples:", len(val_dataset))
print("Batch size:", BATCH_SIZE)
print("Validation ordering: canonical split ordering")

# ------------------------------------------------------------
# 6. RECONSTRUCT EXACT MOBILENETV2
# ------------------------------------------------------------

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

model = mobilenet_v2(
    weights=None,
    num_classes=100,
)

model.features[0][0].stride = (1, 1)

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

model = model.to(device).eval()

total_parameters = sum(
    p.numel() for p in model.parameters()
)

quantizable_layers = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, (nn.Conv2d, nn.Linear))
]

quantizable_weights = sum(
    module.weight.numel()
    for _, module in quantizable_layers
)

assert total_parameters == 2_351_972
assert len(quantizable_layers) == 53
assert quantizable_weights == 2_317_760

print("\nMODEL VERIFICATION")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Recorded best validation accuracy:",
      checkpoint["best_val_accuracy"])
print("Strict state-dict loading:", load_result)
print("Model parameters:", total_parameters)
print("Quantizable layers:", len(quantizable_layers))
print("Quantizable weights:", quantizable_weights)

# ------------------------------------------------------------
# 7. RUN FP32 VALIDATION
# ------------------------------------------------------------

criterion = nn.CrossEntropyLoss()

all_logits = []
all_predictions = []
all_labels = []

total_loss = 0.0
total_correct = 0
total_samples = 0

with torch.inference_mode():

    for batch_idx, (images, labels) in enumerate(val_loader):

        images = images.to(
            device,
            non_blocking=True
        ).float()

        labels = labels.to(
            device,
            non_blocking=True
        )

        logits = model(images)
        loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (predictions == labels).sum().item()
        total_samples += batch_size

        all_logits.append(logits.cpu().numpy())
        all_predictions.append(predictions.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

        if (batch_idx + 1) % 5 == 0 or batch_idx + 1 == len(val_loader):
            print(
                f"Batch {batch_idx + 1}/{len(val_loader)} | "
                f"Samples: {total_samples}/{len(val_dataset)}"
            )

logits = np.concatenate(all_logits).astype(np.float32)
predictions = np.concatenate(all_predictions).astype(np.int64)
labels = np.concatenate(all_labels).astype(np.int64)

accuracy = 100.0 * total_correct / total_samples
average_loss = total_loss / total_samples

assert logits.shape == (5000, 100)
assert predictions.shape == (5000,)
assert labels.shape == (5000,)

# ------------------------------------------------------------
# 8. LOAD AND COMPARE ORIGINAL FP32 REFERENCE
# ------------------------------------------------------------

reference = np.load(REFERENCE_PATH)

ref_logits = reference["logits"]
ref_predictions = reference["predictions"]
ref_labels = reference["labels"]
ref_correct = reference["correct"]
ref_indices = reference["validation_indices"]

assert ref_logits.shape == (5000, 100)
assert np.array_equal(labels, ref_labels)
assert np.array_equal(
    np.asarray(val_indices, dtype=np.int64),
    ref_indices
)

prediction_agreement = np.mean(
    predictions == ref_predictions
) * 100.0

logit_max_abs_difference = float(
    np.max(np.abs(logits - ref_logits))
)

logit_mean_abs_difference = float(
    np.mean(np.abs(logits - ref_logits))
)

reference_accuracy = 100.0 * np.mean(
    ref_predictions == ref_labels
)

reference_correct = int(
    np.sum(ref_correct)
)

print("\n" + "=" * 75)
print("FP32 REPRODUCTION RESULTS")
print("=" * 75)

print(f"Reproduced accuracy:       {accuracy:.4f}%")
print(f"Original reference:        {reference_accuracy:.4f}%")
print(f"Reproduced correct:        {total_correct}/5000")
print(f"Reference correct:         {reference_correct}/5000")
print(f"Reproduced loss:           {average_loss:.8f}")

print("\nREFERENCE AGREEMENT")
print(f"Prediction agreement:      {prediction_agreement:.4f}%")
print(f"Maximum logit difference:  {logit_max_abs_difference:.8e}")
print(f"Mean logit difference:     {logit_mean_abs_difference:.8e}")

# ------------------------------------------------------------
# 9. VALIDATION STATUS
# ------------------------------------------------------------

accuracy_match = abs(accuracy - reference_accuracy) < 1e-6
prediction_match = np.array_equal(predictions, ref_predictions)
label_match = np.array_equal(labels, ref_labels)

print("\n" + "=" * 75)
print("VERIFICATION STATUS")
print("=" * 75)

print("Accuracy matches:", accuracy_match)
print("Predictions match:", prediction_match)
print("Labels match:", label_match)

# ------------------------------------------------------------
# 10. SAVE INDEPENDENT REPRODUCTION ARTIFACTS
# ------------------------------------------------------------

prediction_output = OUTPUT_DIR / "reproduced_fp32_predictions.npz"

np.savez_compressed(
    prediction_output,
    logits=logits,
    predictions=predictions,
    labels=labels,
    correct=(predictions == labels),
    validation_indices=np.asarray(
        val_indices,
        dtype=np.int64
    ),
)

report = {
    "experiment": "JFAMP revised fault-aware allocation",
    "stage": "precheck_0D_fp32_reproduction",
    "checkpoint_sha256": checkpoint_hash,
    "split_sha256": split_hash,
    "reference_prediction_sha256": sha256_file(REFERENCE_PATH),
    "validation_samples": total_samples,
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "model_parameters": int(total_parameters),
    "quantizable_layers": len(quantizable_layers),
    "quantizable_weights": int(quantizable_weights),
    "accuracy_percent": float(accuracy),
    "correct_predictions": int(total_correct),
    "validation_loss": float(average_loss),
    "reference_accuracy_percent": float(reference_accuracy),
    "prediction_agreement_percent": float(prediction_agreement),
    "logit_max_abs_difference": logit_max_abs_difference,
    "logit_mean_abs_difference": logit_mean_abs_difference,
    "accuracy_match": bool(accuracy_match),
    "prediction_match": bool(prediction_match),
    "label_match": bool(label_match),
    "batch_size": BATCH_SIZE,
    "normalization_mean": list(MEAN),
    "normalization_std": list(STD),
    "inference_precision": "FP32",
}

report_path = OUTPUT_DIR / "fp32_reproduction_report.json"

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nSAVED ARTIFACTS")
print(prediction_output)
print(report_path)

print("\n" + "=" * 75)

if accuracy_match and prediction_match and label_match:
    print("SUCCESS — ORIGINAL FP32 BASELINE REPRODUCED EXACTLY.")
elif accuracy_match and label_match:
    print("PARTIAL MATCH — ACCURACY AND LABELS MATCH, "
          "BUT SOME PREDICTIONS DIFFER.")
else:
    print("BASELINE MISMATCH — INVESTIGATION REQUIRED.")

print("=" * 75)

JFAMP — ORIGINAL FP32 BASELINE REPRODUCTION
Device: cuda
GPU: Tesla T4
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130

IDENTITY VERIFICATION
Checkpoint SHA256: 07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a
Split SHA256: e7eaa993c06cb2f5eb9aa8ce111bb8e78ad7a87aa369f04a272e773d23a1039b
Checkpoint identity: VERIFIED
Split identity: VERIFIED

DATASET
Training archive: 50000
Validation samples: 5000
Batch size: 256
Validation ordering: canonical split ordering

MODEL VERIFICATION
Checkpoint epoch: 146
Recorded best validation accuracy: 68.08
Strict state-dict loading: <All keys matched successfully>
Model parameters: 2351972
Quantizable layers: 53
Quantizable weights: 2317760
Batch 5/20 | Samples: 1280/5000
Batch 10/20 | Samples: 2560/5000
Batch 15/20 | Samples: 3840/5000
Batch 20/20 | Samples: 5000/5000

FP32 REPRODUCTION RESULTS
Reproduced accuracy:       68.0600%
Original reference:        68.0600%
Reproduced correct:        3403/5000
Reference correct:         3403/5

In [7]:
# ============================================================
# JFAMP — PRE-CHECK 0E
# AUDIT EXISTING STAGE 4 FAULT CHARACTERIZATION
# ============================================================

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3")
PROJECT = ROOT / "JFAMP_MobileNetV2"

FILES = {
    "stage4c_full_trials":
        PROJECT / "experiments/stage_04_fault_characterization/"
        / "stage4c_full_layer_fault_characterization.csv",

    "stage4c_summary":
        PROJECT / "experiments/stage_04_fault_characterization/"
        / "stage4c_fault_summary.csv",

    "stage4d_model_trials":
        PROJECT / "experiments/stage_04_fault_characterization/"
        / "stage4d_model_level_fault_trials.csv",

    "stage4d_model_summary":
        PROJECT / "experiments/stage_04_fault_characterization/"
        / "stage4d_model_level_summary.csv",
}

# Discover actual paths if the original folder structure differs.
for key, path in list(FILES.items()):
    if not path.exists():
        matches = list(PROJECT.rglob(path.name))
        if matches:
            FILES[key] = matches[0]

print("=" * 90)
print("JFAMP — STAGE 4 FAULT DATA AUDIT")
print("=" * 90)

dataframes = {}

for name, path in FILES.items():

    print("\n" + "-" * 85)
    print(name)
    print("Path:", path)

    if not path.exists():
        print("NOT FOUND")
        continue

    df = pd.read_csv(path)
    dataframes[name] = df

    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nData types:")
    print(df.dtypes.to_string())

    print("\nFirst 5 records:")
    print(df.head().to_string(index=False))

    print("\nMissing values:")
    print(df.isna().sum().to_string())

# ------------------------------------------------------------
# 1. Full layer-level trial coverage
# ------------------------------------------------------------

if "stage4c_full_trials" in dataframes:

    df = dataframes["stage4c_full_trials"]

    print("\n" + "=" * 90)
    print("STAGE 4C — COVERAGE ANALYSIS")
    print("=" * 90)

    for col in df.columns:

        if any(term in col.lower() for term in [
            "layer", "format", "fault", "scenario", "bit",
            "trial", "seed", "status"
        ]):

            print(f"\nColumn: {col}")
            print("Unique values:", df[col].nunique())

            if df[col].nunique() <= 30:
                print(df[col].value_counts(dropna=False).to_string())

    # Detect likely layer and format columns.
    layer_cols = [
        c for c in df.columns
        if "layer" in c.lower()
    ]

    format_cols = [
        c for c in df.columns
        if any(x in c.lower() for x in ["format", "datatype"])
    ]

    fault_cols = [
        c for c in df.columns
        if any(x in c.lower() for x in [
            "fault", "scenario", "injection"
        ])
    ]

    if layer_cols:
        print("\nUnique layer counts:")
        for c in layer_cols:
            print(c, ":", df[c].nunique())

    if format_cols:
        print("\nFormat distributions:")
        for c in format_cols:
            print("\n", c)
            print(df[c].value_counts(dropna=False).to_string())

    if fault_cols:
        print("\nFault scenario distributions:")
        for c in fault_cols:
            print("\n", c)
            print(df[c].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 2. Model-level fault coverage
# ------------------------------------------------------------

if "stage4d_model_trials" in dataframes:

    df = dataframes["stage4d_model_trials"]

    print("\n" + "=" * 90)
    print("STAGE 4D — MODEL-LEVEL COVERAGE")
    print("=" * 90)

    for col in df.columns:

        if any(term in col.lower() for term in [
            "layer", "format", "fault", "scenario",
            "bit", "trial", "seed", "accuracy",
            "baseline", "corruption"
        ]):

            print(f"\nColumn: {col}")
            print("Unique values:", df[col].nunique())

            if df[col].nunique() <= 40:
                print(df[col].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("AUDIT SUMMARY")
print("=" * 90)

for name, df in dataframes.items():
    print(f"{name}: {len(df)} rows, {len(df.columns)} columns")

print("\nAUDIT COMPLETE — NO EXISTING ARTIFACTS MODIFIED")
print("=" * 90)

JFAMP — STAGE 4 FAULT DATA AUDIT

-------------------------------------------------------------------------------------
stage4c_full_trials
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/experiments/stage_04_fault_injection/fault_trials/stage4c_full_layer/stage4c_full_layer_fault_characterization.csv
Rows: 742
Columns: ['format', 'layer', 'fault_group_bits', 'seed', 'weight_count', 'payload_bytes', 'changed_bits', 'selected_bit_positions', 'nonfinite_values', 'finite_weight_mse', 'finite_max_abs_error', 'clean_payload_sha256', 'corrupted_payload_sha256']

Data types:
format                       object
layer                        object
fault_group_bits              int64
seed                          int64
weight_count                  int64
payload_bytes                 int64
changed_bits                  int64
selected_bit_positions       object
nonfinite_values              int64
finite_weight_mse           float64
finite_max_abs_error        float64


In [9]:
# ============================================================
# JFAMP — PRE-CHECK 0F
# DETAILED STAGE 4 COVERAGE REPORT
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100-3/JFAMP_MobileNetV2"
)

FILES = {
    "Stage4C_full": "stage4c_full_layer_fault_characterization.csv",
    "Stage4C_summary": "stage4c_fault_summary.csv",
    "Stage4D_trials": "stage4d_model_level_fault_trials.csv",
    "Stage4D_summary": "stage4d_model_level_summary.csv",
}

for name, filename in FILES.items():

    matches = list(PROJECT.rglob(filename))

    print("\n" + "=" * 95)
    print(name)
    print("=" * 95)

    if not matches:
        print("FILE NOT FOUND:", filename)
        continue

    path = matches[0]
    df = pd.read_csv(path)

    print("Path:", path)
    print("Shape:", df.shape)
    print("\nCOLUMN SCHEMA")

    for col in df.columns:
        print(
            f"{col:45s} | "
            f"dtype={str(df[col].dtype):12s} | "
            f"unique={df[col].nunique():5d} | "
            f"missing={df[col].isna().sum():5d}"
        )

    print("\nUNIQUE VALUES OF CATEGORICAL / IDENTIFIER COLUMNS")

    for col in df.columns:

        if (
            df[col].dtype == "object"
            or df[col].nunique() <= 20
        ):
            print(f"\n{col}:")
            print(df[col].value_counts(dropna=False).head(30).to_string())

    print("\nNUMERIC SUMMARY")

    numeric = df.select_dtypes(include="number")

    if not numeric.empty:
        print(numeric.describe().round(5).to_string())

    print("\nFIRST 3 RECORDS")
    print(df.head(3).to_string(index=False))

print("\n" + "=" * 95)
print("DETAILED STAGE 4 AUDIT COMPLETE — READ ONLY")
print("=" * 95)


Stage4C_full
Path: /kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/experiments/stage_04_fault_injection/fault_trials/stage4c_full_layer/stage4c_full_layer_fault_characterization.csv
Shape: (742, 13)

COLUMN SCHEMA
format                                        | dtype=object       | unique=    7 | missing=    0
layer                                         | dtype=object       | unique=   53 | missing=    0
fault_group_bits                              | dtype=int64        | unique=    2 | missing=    0
seed                                          | dtype=int64        | unique=  742 | missing=    0
weight_count                                  | dtype=int64        | unique=   21 | missing=    0
payload_bytes                                 | dtype=int64        | unique=   49 | missing=    0
changed_bits                                  | dtype=int64        | unique=    2 | missing=    0
selected_bit_positions                        | dtype=object       | unique=

In [10]:
# ============================================================
# JFAMP — PRE-CHECK 0G
# STAGE 4D CLEAN REFERENCE AND TRIAL CONSISTENCY
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100-3/JFAMP_MobileNetV2"
)

path = next(PROJECT.rglob("stage4d_model_level_fault_trials.csv"))

df = pd.read_csv(path)

print("=" * 90)
print("STAGE 4D — CLEAN REFERENCE AND TRIAL CONSISTENCY")
print("=" * 90)

# ------------------------------------------------------------
# 1. Verify the grouping and realization counts
# ------------------------------------------------------------

group_cols = ["layer", "format", "fault_bits"]

counts = (
    df.groupby(group_cols)
      .agg(
          trials=("realization", "count"),
          unique_realizations=("realization", "nunique"),
          min_accuracy=("accuracy_percent", "min"),
          max_accuracy=("accuracy_percent", "max"),
          mean_accuracy=("accuracy_percent", "mean"),
          mean_delta=("accuracy_delta_vs_clean_pp", "mean"),
          worst_delta=("accuracy_delta_vs_clean_pp", "min"),
          best_delta=("accuracy_delta_vs_clean_pp", "max"),
      )
      .reset_index()
)

print("\nGROUP COVERAGE")
print(counts.to_string(index=False))

# ------------------------------------------------------------
# 2. Inspect the five-layer clean accuracy references
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("INFERRED CLEAN ACCURACY PER LAYER / FORMAT")
print("=" * 90)

# accuracy_delta_vs_clean_pp is expected to equal:
# faulted accuracy - clean accuracy.
# Therefore clean accuracy = faulted accuracy - delta.

df["inferred_clean_accuracy"] = (
    df["accuracy_percent"] -
    df["accuracy_delta_vs_clean_pp"]
)

clean = (
    df.groupby(["layer", "format"])
      ["inferred_clean_accuracy"]
      .agg(["mean", "std", "min", "max", "count"])
      .reset_index()
)

print(clean.to_string(index=False))

# ------------------------------------------------------------
# 3. Check whether the clean reference is consistent
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("CLEAN REFERENCE CONSISTENCY")
print("=" * 90)

format_consistency = (
    df.groupby("format")["inferred_clean_accuracy"]
      .agg(["mean", "std", "min", "max", "count"])
)

print(format_consistency.to_string())

# ------------------------------------------------------------
# 4. Compare with the canonical FP32 baseline
# ------------------------------------------------------------

FP32_REFERENCE = 68.06

print("\n" + "=" * 90)
print("COMPARISON WITH CANONICAL FP32 REFERENCE")
print("=" * 90)

format_comparison = (
    df.groupby("format")
      ["inferred_clean_accuracy"]
      .mean()
      .reset_index(name="stage4d_inferred_clean_accuracy")
)

format_comparison["delta_vs_fp32_reference_pp"] = (
    format_comparison["stage4d_inferred_clean_accuracy"]
    - FP32_REFERENCE
)

print(format_comparison.to_string(index=False))

# ------------------------------------------------------------
# 5. Inspect suspicious or extreme trials
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("EXTREME ACCURACY-DEGRADATION TRIALS")
print("=" * 90)

extreme = df.nsmallest(
    15,
    "accuracy_delta_vs_clean_pp"
)[[
    "layer",
    "format",
    "fault_bits",
    "realization",
    "accuracy_percent",
    "accuracy_delta_vs_clean_pp",
    "loss",
    "weight_max_abs_error",
    "verified_bit_flips",
]]

print(extreme.to_string(index=False))

# ------------------------------------------------------------
# 6. Final checks
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL AUDIT")
print("=" * 90)

print("Total trials:", len(df))
print("Unique layers:", df["layer"].nunique())
print("Unique formats:", df["format"].nunique())
print("Fault scenarios:", sorted(df["fault_bits"].unique()))
print("All statuses evaluated:", (df["status"] == "evaluated").all())
print("All bit flips verified:",
      (df["verified_bit_flips"] == df["fault_bits"]).all())

print("\nAUDIT COMPLETE — READ ONLY")
print("=" * 90)

STAGE 4D — CLEAN REFERENCE AND TRIAL CONSISTENCY

GROUP COVERAGE
              layer     format  fault_bits  trials  unique_realizations  min_accuracy  max_accuracy  mean_accuracy    mean_delta  worst_delta  best_delta
features.1.conv.0.0       BF16           1       5                    5         23.32         68.10         59.124 -8.976000e+00       -44.78        0.00
features.1.conv.0.0       BF16           8       5                    5          1.10         67.84         52.316 -1.578400e+01       -67.00       -0.26
features.1.conv.0.0   E4M11_SN           1       5                    5         66.60         68.06         67.768 -2.920000e-01        -1.46        0.00
features.1.conv.0.0   E4M11_SN           8       5                    5          3.30         68.06         55.100 -1.296000e+01       -64.76        0.00
features.1.conv.0.0       FP16           1       5                    5         68.06         68.14         68.080  2.000000e-02         0.00        0.08
features.1.

In [11]:
# ============================================================
# JFAMP — PRE-CHECK 0H
# EXTRACT ORIGINAL STAGE 4D IMPLEMENTATION
# ============================================================

from pathlib import Path
import json

NOTEBOOK_PATH = Path(
    "/kaggle/input/notebooks/crahulnaidu/"
    "mobile-net-100-3/__notebook__.ipynb"
)

assert NOTEBOOK_PATH.exists(), f"Missing: {NOTEBOOK_PATH}"

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "stage4d",
    "model_level_fault",
    "inject_bit_flips",
    "decode_serialized_payload",
    "encode_weights",
    "decode_weights",
    "accuracy_delta_vs_clean_pp",
    "clean_payload_sha256",
    "realization",
    "fault_bits",
]

print("=" * 100)
print("ORIGINAL STAGE 4D SOURCE INSPECTION")
print("=" * 100)

cells = notebook.get("cells", [])

matched_cells = []

for idx, cell in enumerate(cells):

    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    matched = [
        keyword for keyword in keywords
        if keyword.lower() in source.lower()
    ]

    if matched:
        matched_cells.append((idx, matched, source))

print("Total notebook cells:", len(cells))
print("Matched code cells:", len(matched_cells))

for idx, matched, source in matched_cells:

    print("\n" + "=" * 100)
    print(f"CELL {idx}")
    print("MATCHES:", matched)
    print("=" * 100)

    print(source[:25000])

    if len(source) > 25000:
        print("\n... SOURCE TRUNCATED ...")

print("\n" + "=" * 100)
print("SOURCE EXTRACTION COMPLETE — NO FILES MODIFIED")
print("=" * 100)

ORIGINAL STAGE 4D SOURCE INSPECTION
Total notebook cells: 52
Matched code cells: 26

CELL 6
MATCHES: ['encode_weights', 'decode_weights']

# ================================================================
# JFAMP — STAGE 4B: SERIALIZED PAYLOAD ROUND-TRIP VALIDATION
# ================================================================

import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torchvision.models import mobilenet_v2

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")
CHECKPOINT_PATH = ROOT / "checkpoints/mobilenetv2_cifar100_best.pth"
STAGE2 = ROOT / "experiments/stage_02_baseline"
STAGE4 = ROOT / "experiments/stage_04_fault_injection"

REFERENCE_CSV = STAGE2 / "codec_validation_per_layer.csv"
REFERENCE_JSON = STAGE2 / "codec_validation_summary.json"

FORMATS = [
    "FP16", "BF16", "E4M11_SN",
    "FP8_E4M3FN", "FP8_E5M2", "INT8", "INT4"
]

print("=" * 72)
print("JFAMP — STAGE 4B: SERIALIZED PAYLOAD VA

In [16]:

# =====================================================================
# JFAMP — STAGE 4D EXACT TRIAL REPRODUCTION CHECK
#
# Reproduce:
# Layer       : features.2.conv.2
# Format      : FP16
# Fault       : 1 bit
# Realization : 1
#
# Original Stage 4D seed formula and codec are preserved.
# =====================================================================

import copy
import hashlib
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ---------------------------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ---------------------------------------------------------------------

ROOT = Path("/kaggle/working/JFAMP_MobileNetV2")

STAGE4D_CSV = (
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3")
    / "JFAMP_MobileNetV2/experiments/stage_04_fault_injection"
    / "results/stage4d/stage4d_model_level_fault_trials.csv"
)

CHECKPOINT_PATH = (
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100")
    / "JFAMP_MobileNetV2/checkpoints/mobilenetv2_cifar100_best.pth"
)

REFERENCE_PATH = (
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3")
    / "JFAMP_MobileNetV2/experiments/stage_02_baseline"
    / "fp32_reference_predictions.npz"
)

OUT = (
    ROOT / "experiments/revised_jfamp_fault_aware"
    / "reproduction_check"
)
OUT.mkdir(parents=True, exist_ok=True)

LAYER = "features.2.conv.2"
FORMAT = "FP16"
FAULT_BITS = 1
REALIZATION = 1

MASTER_SEED = 20261114

# IMPORTANT: these are the original Stage 4D layer and format orders.
ORIGINAL_LAYERS = [
    "features.2.conv.2",
    "features.2.conv.1.0",
    "features.4.conv.2",
    "features.7.conv.2",
    "features.1.conv.0.0"
]

FORMATS = [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4"
]

layer_index = ORIGINAL_LAYERS.index(LAYER)
format_index = FORMATS.index(FORMAT)

SEED = (
    MASTER_SEED
    + layer_index * 100000
    + format_index * 10000
    + FAULT_BITS * 100
    + REALIZATION
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ---------------------------------------------------------------------
# 2. INPUT VALIDATION
# ---------------------------------------------------------------------

assert "model" in globals(), "Model is missing."
assert "val_loader" in globals(), "Validation loader is missing."
assert callable(encode_weights), "Stage 4B encoder is missing."
assert callable(decode_weights), "Stage 4B decoder is missing."

for path in [CHECKPOINT_PATH, REFERENCE_PATH, STAGE4D_CSV]:
    assert path.is_file(), f"Missing input: {path}"

print("=" * 76)
print("JFAMP — EXACT STAGE 4D TRIAL REPRODUCTION")
print("=" * 76)

print("Device:", DEVICE)
print("Layer:", LAYER)
print("Format:", FORMAT)
print("Fault bits:", FAULT_BITS)
print("Realization:", REALIZATION)
print("Original layer index:", layer_index)
print("Original format index:", format_index)
print("Derived seed:", SEED)

# ---------------------------------------------------------------------
# 3. LOAD ARCHIVED TRIAL
# ---------------------------------------------------------------------

archived = pd.read_csv(STAGE4D_CSV)

target = archived[
    (archived["layer"] == LAYER)
    & (archived["format"] == FORMAT)
    & (archived["fault_bits"] == FAULT_BITS)
    & (archived["realization"] == REALIZATION)
]

assert len(target) == 1, (
    f"Expected exactly one archived trial, found {len(target)}"
)

archived_trial = target.iloc[0]

print("\nArchived trial:")
print(
    archived_trial[
        [
            "seed",
            "status",
            "accuracy_percent",
            "accuracy_delta_vs_clean_pp",
            "verified_bit_flips",
            "payload_bytes",
            "clean_payload_sha256",
            "corrupted_payload_sha256"
        ]
    ].to_string()
)

assert int(archived_trial["seed"]) == SEED
assert archived_trial["status"] == "evaluated"

# ---------------------------------------------------------------------
# 4. LOAD ORIGINAL CHECKPOINT AND REFERENCE
# ---------------------------------------------------------------------

reference = np.load(REFERENCE_PATH)

reference_predictions = reference["predictions"].astype(np.int64)
reference_labels = reference["labels"].astype(np.int64)

assert len(reference_predictions) == 5000
assert len(reference_labels) == 5000
assert len(val_loader.dataset) == 5000

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

assert "model_state_dict" in checkpoint

base_model = copy.deepcopy(model).cpu()

base_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

base_model.eval()

named_parameters = dict(base_model.named_parameters())

parameter_name = LAYER + ".weight"

assert parameter_name in named_parameters

original_weight = (
    named_parameters[parameter_name]
    .detach()
    .cpu()
    .clone()
)

original_shape = tuple(original_weight.shape)

# ---------------------------------------------------------------------
# 5. EXACT STAGE 4D EVALUATION FUNCTION
# ---------------------------------------------------------------------

@torch.inference_mode()
def evaluate_model(net):

    net = net.to(DEVICE).eval()

    predictions = []
    labels = []
    total_loss = 0.0
    total = 0

    criterion = torch.nn.CrossEntropyLoss(reduction="sum")

    for images, targets in val_loader:

        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        logits = net(images)

        total_loss += criterion(logits, targets).item()

        predictions.extend(
            logits.argmax(dim=1).cpu().numpy().tolist()
        )

        labels.extend(
            targets.cpu().numpy().tolist()
        )

        total += targets.numel()

    net.cpu()

    predictions = np.asarray(predictions, dtype=np.int64)
    labels = np.asarray(labels, dtype=np.int64)

    assert total == 5000
    assert np.array_equal(labels, reference_labels)

    return {
        "predictions": predictions,
        "correct": predictions == labels,
        "accuracy": float(
            np.mean(predictions == labels) * 100.0
        ),
        "loss": float(total_loss / total)
    }


def sha256_bytes(array):
    return hashlib.sha256(
        np.ascontiguousarray(array).tobytes()
    ).hexdigest()


def decode_serialized_payload(payload, metadata, fmt):

    decoded = decode_weights(
        payload.copy(),
        copy.deepcopy(metadata),
        fmt
    )

    if isinstance(decoded, tuple):
        decoded = decoded[0]

    if isinstance(decoded, np.ndarray):
        decoded = torch.from_numpy(decoded.copy())

    if not isinstance(decoded, torch.Tensor):
        decoded = torch.as_tensor(decoded)

    return decoded.detach().cpu().float()


# ---------------------------------------------------------------------
# 6. VERIFY FP32 BASELINE
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("A. FP32 BASELINE")
print("-" * 76)

fp32_result = evaluate_model(base_model)

fp32_prediction_match = np.array_equal(
    fp32_result["predictions"],
    reference_predictions
)

assert fp32_prediction_match, (
    "FP32 predictions do not match the canonical reference."
)

print(f"Accuracy: {fp32_result['accuracy']:.4f}%")
print(f"Loss: {fp32_result['loss']:.8f}")
print("Predictions exactly match reference:", fp32_prediction_match)

# ---------------------------------------------------------------------
# 7. ENCODE AND DECODE CLEAN FP16 PAYLOAD
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("B. CLEAN FP16 SERIALIZATION")
print("-" * 76)

encoded = encode_weights(original_weight, FORMAT)

assert isinstance(encoded, tuple)
assert len(encoded) == 2

clean_payload, metadata = encoded

assert isinstance(clean_payload, np.ndarray)
assert clean_payload.dtype == np.uint8
assert isinstance(metadata, dict)
assert tuple(metadata["shape"]) == original_shape

clean_payload = clean_payload.copy()
metadata = copy.deepcopy(metadata)

clean_hash = sha256_bytes(clean_payload)

clean_weight = decode_serialized_payload(
    clean_payload,
    metadata,
    FORMAT
)

assert tuple(clean_weight.shape) == original_shape
assert torch.isfinite(clean_weight).all()

clean_model = copy.deepcopy(base_model)

dict(clean_model.named_parameters())[
    parameter_name
].data.copy_(clean_weight)

clean_result = evaluate_model(clean_model)

print("Payload bytes:", clean_payload.nbytes)
print("Clean payload SHA256:", clean_hash)
print(f"Clean accuracy: {clean_result['accuracy']:.4f}%")
print(f"Clean loss: {clean_result['loss']:.8f}")

# ---------------------------------------------------------------------
# 8. COMPARE CLEAN MODEL WITH ARCHIVED STAGE 4D RECORD
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("C. CLEAN REFERENCE COMPARISON")
print("-" * 76)

# Stage 4D clean accuracy is stored in its trial delta:
# delta = fault accuracy - clean accuracy.
# We recover clean accuracy from the archived row.

archived_clean_accuracy = (
    float(archived_trial["accuracy_percent"])
    - float(archived_trial["accuracy_delta_vs_clean_pp"])
)

print(
    f"Reproduced clean accuracy: "
    f"{clean_result['accuracy']:.4f}%"
)

print(
    f"Archived inferred clean accuracy: "
    f"{archived_clean_accuracy:.4f}%"
)

print(
    "Clean payload hash matches:",
    clean_hash == archived_trial["clean_payload_sha256"]
)

print(
    "Clean accuracy matches:",
    np.isclose(
        clean_result["accuracy"],
        archived_clean_accuracy,
        atol=1e-6
    )
)

# ---------------------------------------------------------------------
# 9. REGENERATE EXACT ORIGINAL RANDOM BIT FLIP
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("D. ORIGINAL BIT-FLIP REGENERATION")
print("-" * 76)

rng = np.random.default_rng(SEED)

total_payload_bits = clean_payload.size * 8

selected_bits = rng.choice(
    total_payload_bits,
    size=FAULT_BITS,
    replace=False
)

selected_bit = int(selected_bits[0])

byte_index = selected_bit // 8
bit_index = selected_bit % 8

corrupted_payload = clean_payload.copy()

corrupted_payload[byte_index] ^= np.uint8(
    1 << bit_index
)

xor_payload = np.bitwise_xor(
    clean_payload,
    corrupted_payload
)

verified_flips = int(
    np.unpackbits(xor_payload).sum()
)

corrupted_hash = sha256_bytes(corrupted_payload)

assert verified_flips == FAULT_BITS
assert corrupted_hash != clean_hash

print("Selected bit position:", selected_bit)
print("Byte index:", byte_index)
print("Bit within byte:", bit_index)
print("Verified bit flips:", verified_flips)
print("Corrupted payload SHA256:", corrupted_hash)

print(
    "Corrupted hash matches archive:",
    corrupted_hash == archived_trial["corrupted_payload_sha256"]
)

# ---------------------------------------------------------------------
# 10. DECODE CORRUPTED PAYLOAD
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("E. CORRUPTED PAYLOAD DECODING")
print("-" * 76)

corrupted_weight = decode_serialized_payload(
    corrupted_payload,
    metadata,
    FORMAT
)

assert tuple(corrupted_weight.shape) == original_shape

finite_decode = bool(torch.isfinite(corrupted_weight).all())

print("Decoded shape:", tuple(corrupted_weight.shape))
print("All decoded weights finite:", finite_decode)

assert finite_decode, (
    "Archived trial was evaluated, but reproduction decoded non-finite weights."
)

weight_difference = corrupted_weight - clean_weight

weight_mse = float(
    torch.mean(weight_difference ** 2).item()
)

weight_max_error = float(
    torch.max(torch.abs(weight_difference)).item()
)

print(f"Weight MSE vs clean: {weight_mse:.10e}")
print(f"Maximum weight error: {weight_max_error:.10e}")

# ---------------------------------------------------------------------
# 11. RECONSTRUCT CORRUPTED MODEL AND EVALUATE
# ---------------------------------------------------------------------

print("\n" + "-" * 76)
print("F. FULL MODEL FAULT EVALUATION")
print("-" * 76)

fault_model = copy.deepcopy(base_model)

dict(fault_model.named_parameters())[
    parameter_name
].data.copy_(corrupted_weight)

fault_result = evaluate_model(fault_model)

accuracy_delta = (
    fault_result["accuracy"] - clean_result["accuracy"]
)

prediction_changes = int(
    np.sum(
        fault_result["predictions"]
        != clean_result["predictions"]
    )
)

clean_correct = clean_result["correct"]
fault_correct = fault_result["correct"]

correct_to_wrong = int(
    np.sum(clean_correct & ~fault_correct)
)

wrong_to_correct = int(
    np.sum(~clean_correct & fault_correct)
)

print(f"Faulted accuracy: {fault_result['accuracy']:.4f}%")
print(f"Faulted loss: {fault_result['loss']:.8f}")
print(f"Accuracy delta vs clean: {accuracy_delta:+.4f} pp")
print("Prediction changes:", prediction_changes)
print("Correct -> wrong:", correct_to_wrong)
print("Wrong -> correct:", wrong_to_correct)

# ---------------------------------------------------------------------
# 12. ARCHIVED VS REPRODUCED COMPARISON
# ---------------------------------------------------------------------

print("\n" + "=" * 76)
print("G. FINAL REPRODUCTION COMPARISON")
print("=" * 76)

checks = {
    "Seed matches": (
        SEED == int(archived_trial["seed"])
    ),
    "FP32 reference predictions match": (
        fp32_prediction_match
    ),
    "Clean payload hash matches": (
        clean_hash == archived_trial["clean_payload_sha256"]
    ),
    "Clean accuracy matches": (
        np.isclose(
            clean_result["accuracy"],
            archived_clean_accuracy,
            atol=1e-6
        )
    ),
    "Verified bit count matches": (
        verified_flips == int(archived_trial["verified_bit_flips"])
    ),
    "Corrupted payload hash matches": (
        corrupted_hash == archived_trial["corrupted_payload_sha256"]
    ),
    "Faulted accuracy matches": (
        np.isclose(
            fault_result["accuracy"],
            float(archived_trial["accuracy_percent"]),
            atol=1e-6
        )
    ),
    "Accuracy delta matches": (
        np.isclose(
            accuracy_delta,
            float(archived_trial["accuracy_delta_vs_clean_pp"]),
            atol=1e-6
        )
    ),
    "Prediction changes match": (
        prediction_changes == int(
            archived_trial["prediction_changes_vs_clean"]
        )
    ),
    "Correct-to-wrong matches": (
        correct_to_wrong == int(
            archived_trial["correct_to_wrong"]
        )
    ),
    "Wrong-to-correct matches": (
        wrong_to_correct == int(
            archived_trial["wrong_to_correct"]
        )
    )
}

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")

all_passed = all(checks.values())

# Save machine-readable reproduction report.
report = {
    "experiment": "JFAMP_Stage4D_exact_trial_reproduction",
    "layer": LAYER,
    "format": FORMAT,
    "fault_bits": FAULT_BITS,
    "realization": REALIZATION,
    "seed": SEED,
    "selected_bit_position": selected_bit,
    "byte_index": byte_index,
    "bit_index": bit_index,
    "fp32_accuracy": fp32_result["accuracy"],
    "clean_accuracy": clean_result["accuracy"],
    "fault_accuracy": fault_result["accuracy"],
    "accuracy_delta_vs_clean_pp": accuracy_delta,
    "prediction_changes": prediction_changes,
    "correct_to_wrong": correct_to_wrong,
    "wrong_to_correct": wrong_to_correct,
    "clean_payload_sha256": clean_hash,
    "corrupted_payload_sha256": corrupted_hash,
    "weight_mse": weight_mse,
    "weight_max_abs_error": weight_max_error,
    "checks": checks,
    "all_checks_passed": all_passed
}

report_path = OUT / "stage4d_exact_trial_reproduction.json"

import json

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("\nReport saved:", report_path)

print("\n" + "=" * 76)

if all_passed:
    print("[SUCCESS] Exact Stage 4D trial reproduced.")
    print("We can proceed to the expanded layer-wise fault campaign.")
else:
    print("[STOP] Reproduction mismatch detected.")
    print("Do not launch the expanded experiment yet.")
    print("Inspect failed checks before proceeding.")

print("=" * 76)


JFAMP — EXACT STAGE 4D TRIAL REPRODUCTION
Device: cuda
Layer: features.2.conv.2
Format: FP16
Fault bits: 1
Realization: 1
Original layer index: 0
Original format index: 0
Derived seed: 20261215

Archived trial:
seed                                                                   20261215
status                                                                evaluated
accuracy_percent                                                          68.06
accuracy_delta_vs_clean_pp                                                  0.0
verified_bit_flips                                                            1
payload_bytes                                                              4608
clean_payload_sha256          283c49b585ae3eca830c5c386fc903dd8f9b504788802e...
corrupted_payload_sha256      4ad5b499d2f124fa3e0013ddba3ec7bee6f5796cd41200...

----------------------------------------------------------------------------
A. FP32 BASELINE
------------------------------------------------------

TypeError: Object of type bool is not JSON serializable

In [13]:

# ============================================================
# RESTORE ORIGINAL STAGE 4B FP16 CODEC
# Exact FP16 branch from the original JFAMP implementation
# ============================================================

import numpy as np
import torch

def encode_weights(weights, fmt):

    if fmt != "FP16":
        raise ValueError(
            "This reproduction check currently supports FP16 only."
        )

    x = np.asarray(
        weights,
        dtype=np.float32
    ).reshape(-1)

    t = torch.from_numpy(
        x.copy()
    ).to(torch.float16)

    payload = t.view(torch.uint8).numpy().copy()

    metadata = {
        "shape": list(weights.shape)
    }

    return payload, metadata


def decode_weights(payload, metadata, fmt):

    if fmt != "FP16":
        raise ValueError(
            "This reproduction check currently supports FP16 only."
        )

    shape = tuple(metadata["shape"])

    decoded = torch.from_numpy(
        payload.copy()
    ).view(torch.float16).float().numpy()

    return decoded.reshape(shape)


assert callable(encode_weights)
assert callable(decode_weights)

print("Original FP16 encoder restored.")
print("Original FP16 decoder restored.")
print("Ready for Stage 4D reproduction.")


Original FP16 encoder restored.
Original FP16 decoder restored.
Ready for Stage 4D reproduction.


In [15]:

# ============================================================
# JFAMP — LOCATE CANONICAL FP32 REFERENCE FILES
# Read-only path inspection
# ============================================================

from pathlib import Path

SEARCH_ROOTS = [
    Path("/kaggle/working/JFAMP_MobileNetV2"),
    Path("/kaggle/input/notebooks/crahulnaidu")
]

patterns = [
    "fp32_reference_predictions.npz",
    "reproduced_fp32_predictions.npz",
    "stage4d_model_level_fault_trials.csv"
]

for pattern in patterns:

    print("\n" + "=" * 75)
    print("SEARCHING FOR:", pattern)
    print("=" * 75)

    matches = []

    for root in SEARCH_ROOTS:

        if root.exists():
            matches.extend(root.rglob(pattern))

    matches = sorted(set(matches))

    if matches:
        for path in matches:
            print(path)
            print("Size:", round(path.stat().st_size / 1024, 2), "KB")
    else:
        print("NOT FOUND")



SEARCHING FOR: fp32_reference_predictions.npz
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_02_baseline/fp32_reference_predictions.npz
Size: 1841.72 KB
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/experiments/stage_02_baseline/fp32_reference_predictions.npz
Size: 1841.72 KB
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/results/fp32_reference_predictions.npz
Size: 15.73 KB

SEARCHING FOR: reproduced_fp32_predictions.npz
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/precheck_fp32_reproduction/reproduced_fp32_predictions.npz
Size: 1841.72 KB

SEARCHING FOR: stage4d_model_level_fault_trials.csv
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2/experiments/stage_04_fault_injection/results/stage4d/stage4d_model_level_fault_trials.csv
Size: 96.55 KB


In [17]:
import json
import numpy as np
import torch
from pathlib import Path

def json_safe(obj):
    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()

    if isinstance(obj, Path):
        return str(obj)

    raise TypeError(
        f"Unsupported JSON type: {type(obj).__name__}"
    )

with open(report_path, "w") as f:
    json.dump(
        report,
        f,
        indent=2,
        default=json_safe
    )

print("Report successfully saved:", report_path)

# Verify that the saved report is readable.
with open(report_path, "r") as f:
    saved_report = json.load(f)

print("JSON verification: PASS")
print("Report size:", Path(report_path).stat().st_size, "bytes")

Report successfully saved: /kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/reproduction_check/stage4d_exact_trial_reproduction.json
JSON verification: PASS
Report size: 1175 bytes


In [18]:
from pathlib import Path
import re

ROOTS = [
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2"),
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2"),
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2"),
]

patterns = [
    "def encode_weights",
    "def decode_weights",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
]

extensions = {".py", ".ipynb"}

print("=" * 85)
print("JFAMP — ORIGINAL DATATYPE CODEC SOURCE AUDIT")
print("=" * 85)

matches = 0

for root in ROOTS:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        try:
            content = path.read_text(errors="ignore")
        except Exception:
            continue

        found = [p for p in patterns if p in content]

        if not found:
            continue

        matches += 1
        print(f"\nFILE: {path}")
        print("MATCHES:", ", ".join(found))

        lines = content.splitlines()

        if path.suffix.lower() == ".py":
            for i, line in enumerate(lines):
                if "def encode_weights" in line or "def decode_weights" in line:
                    start = max(0, i - 3)
                    end = min(len(lines), i + 100)
                    print(f"\n--- Lines {start + 1}–{end} ---")
                    for j in range(start, end):
                        print(f"{j + 1:4}: {lines[j]}")

        else:
            # Notebook source is JSON; print matching code-cell fragments.
            for i, line in enumerate(lines):
                if any(p in line for p in patterns):
                    start = max(0, i - 2)
                    end = min(len(lines), i + 15)
                    print(f"\n--- Notebook JSON lines {start + 1}–{end} ---")
                    for j in range(start, end):
                        print(f"{j + 1:4}: {lines[j][:500]}")

print("\n" + "=" * 85)
print("Files containing relevant codec references:", matches)
print("Share the output before proceeding.")

JFAMP — ORIGINAL DATATYPE CODEC SOURCE AUDIT

Files containing relevant codec references: 0
Share the output before proceeding.


In [19]:
from pathlib import Path

ROOTS = [
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2"),
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2"),
    Path("/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/JFAMP_MobileNetV2"),
]

for root in ROOTS:
    print("\n" + "=" * 90)
    print("ROOT:", root)
    print("EXISTS:", root.exists())

    if not root.exists():
        continue

    files = [p for p in root.rglob("*") if p.is_file()]

    print("TOTAL FILES:", len(files))

    print("\nRELEVANT SOURCE / NOTEBOOK FILES:")
    for p in files:
        name = p.name.lower()
        if (
            p.suffix.lower() in {
                ".py", ".ipynb", ".txt", ".md",
                ".json", ".yaml", ".yml"
            }
            and any(k in str(p).lower() for k in [
                "stage_04", "stage4", "quant", "codec",
                "fault", "precision", "format", "experiment"
            ])
        ):
            print(p)

    print("\nALL PYTHON AND NOTEBOOK FILES:")
    for p in files:
        if p.suffix.lower() in {".py", ".ipynb"}:
            print(p)

print("\nDirectory inspection complete.")


ROOT: /kaggle/input/notebooks/crahulnaidu/mobile-net-100/JFAMP_MobileNetV2
EXISTS: True
TOTAL FILES: 12

RELEVANT SOURCE / NOTEBOOK FILES:

ALL PYTHON AND NOTEBOOK FILES:

ROOT: /kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2
EXISTS: True
TOTAL FILES: 22

RELEVANT SOURCE / NOTEBOOK FILES:
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_01_layer_inventory/inventory_summary.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_01_layer_inventory/layer_inventory.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_02_baseline/codec_validation_summary.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_02_baseline/e4m11_codec_validation.json
/kaggle/input/notebooks/crahulnaidu/mobile-net-100-2/JFAMP_MobileNetV2/experiments/stage_02_baseline/experiment_identity.json
/kaggle/input/notebooks/crahulnaidu/mobi

In [20]:
import json
from pathlib import Path

ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/"
    "JFAMP_MobileNetV2"
)

FILES = [
    "experiments/stage_04_fault_injection/manifests/stage4_config.json",
    "experiments/stage_04_fault_injection/manifests/stage4b_codec_audit.json",
    "experiments/stage_04_fault_injection/manifests/stage4b_payload_hashes.json",
    "experiments/stage_02_baseline/codec_validation_summary.json",
    "experiments/stage_02_baseline/e4m11_codec_validation.json",
    "experiments/stage_02_baseline/numerical_format_probe.json",
    "experiments/stage_02_baseline/quantizer_runtime_audit.json",
]

for relative_path in FILES:
    path = ROOT / relative_path

    print("\n" + "=" * 100)
    print("FILE:", relative_path)
    print("=" * 100)

    if not path.exists():
        print("NOT FOUND")
        continue

    try:
        with open(path, "r") as f:
            data = json.load(f)

        print(json.dumps(data, indent=2, default=str)[:18000])

    except Exception as e:
        print("READ ERROR:", repr(e))

print("\nCodec artifact inspection complete.")


FILE: experiments/stage_04_fault_injection/manifests/stage4_config.json
{
  "experiment": "JFAMP_MobileNetV2_CIFAR100",
  "stage": "04_encoded_weight_fault_injection",
  "architecture": "MobileNetV2",
  "dataset": "CIFAR-100",
  "quantizable_layers": 53,
  "formats": [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4"
  ],
  "fault_groups": [
    1,
    8
  ],
  "fault_injection_target": "Serialized quantized weight payload",
  "inference": "Decode corrupted payload and reconstruct FP32 weights",
  "status": "workspace_prepared"
}

FILE: experiments/stage_04_fault_injection/manifests/stage4b_codec_audit.json
{
  "stage": "04B_clean_serialized_payload_roundtrip",
  "checkpoint": "mobilenetv2_cifar100_best.pth",
  "checkpoint_sha256": "07c3c343451dcef593efc9172b9e07194b72f7c8c635b115f03104d7a8b2ff0a",
  "codec_source": "Original unified Stage 2 Cell 12 implementation",
  "formats": [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",

In [21]:
import hashlib
import json
import numpy as np
import torch
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/"
    "JFAMP_MobileNetV2"
)

HASH_PATH = (
    ROOT / "experiments/stage_04_fault_injection/"
    "manifests/stage4b_payload_hashes.json"
)

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/"
    "experiments/revised_jfamp_fault_aware/codec_validation"
)
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Original E4M11-SN implementation
# ------------------------------------------------------------

EXP_BITS = 4
FRAC_BITS = 11
BIAS = 7

EXP_MASK = 0xF
FRAC_MASK = (1 << FRAC_BITS) - 1

MIN_NORMAL = 2.0 ** (1 - BIAS)
MIN_SUBNORMAL = 2.0 ** (1 - BIAS - FRAC_BITS)
MAX_FINITE = (
    (2.0 - 2.0 ** (-FRAC_BITS))
    * (2.0 ** (14 - BIAS))
)

def encode_e4m11(values):
    x = np.asarray(values, dtype=np.float64)
    flat = x.reshape(-1)
    encoded = np.zeros(flat.shape, dtype=np.uint16)

    for i, value in enumerate(flat):
        sign = 1 if np.signbit(value) else 0
        sign_bits = sign << 15
        magnitude = abs(value)

        if np.isnan(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS) | 1
            continue

        if np.isinf(value):
            encoded[i] = sign_bits | (15 << FRAC_BITS)
            continue

        if magnitude == 0:
            encoded[i] = sign_bits
            continue

        if magnitude < MIN_NORMAL:
            fraction = int(np.rint(magnitude / MIN_SUBNORMAL))

            if fraction == 0:
                encoded[i] = sign_bits
            elif fraction >= (1 << FRAC_BITS):
                encoded[i] = sign_bits | (1 << FRAC_BITS)
            else:
                encoded[i] = sign_bits | fraction

            continue

        exponent = int(np.floor(np.log2(magnitude)))
        exponent_field = exponent + BIAS

        significand = magnitude / (2.0 ** exponent)
        fraction = int(
            np.rint((significand - 1.0) * (1 << FRAC_BITS))
        )

        if fraction >= (1 << FRAC_BITS):
            fraction = 0
            exponent_field += 1

        if exponent_field >= 15:
            encoded[i] = sign_bits | (15 << FRAC_BITS)
        else:
            encoded[i] = (
                sign_bits
                | (exponent_field << FRAC_BITS)
                | fraction
            )

    return encoded.reshape(x.shape)


def decode_e4m11(encoded):
    bits = np.asarray(encoded, dtype=np.uint16)
    flat = bits.reshape(-1)
    decoded = np.zeros(flat.shape, dtype=np.float64)

    for i, word in enumerate(flat):
        sign = -1.0 if (int(word) >> 15) & 1 else 1.0

        exponent_field = (int(word) >> FRAC_BITS) & EXP_MASK
        fraction = int(word) & FRAC_MASK

        if exponent_field == 15:
            if fraction == 0:
                decoded[i] = sign * np.inf
            else:
                decoded[i] = np.nan
            continue

        if exponent_field == 0:
            magnitude = fraction * MIN_SUBNORMAL
        else:
            exponent = exponent_field - BIAS
            significand = 1.0 + fraction / (1 << FRAC_BITS)
            magnitude = significand * (2.0 ** exponent)

        decoded[i] = sign * magnitude

    return decoded.reshape(bits.shape)


def encode_weights(weights, fmt):
    if fmt != "E4M11_SN":
        raise ValueError(f"Unsupported format: {fmt}")

    if isinstance(weights, torch.Tensor):
        original = weights.detach().cpu().numpy()
    else:
        original = np.asarray(weights)

    encoded = encode_e4m11(original)

    payload = encoded.view(np.uint8).reshape(-1).copy()

    metadata = {
        "shape": list(original.shape)
    }

    return payload, metadata


def decode_weights(payload, metadata, fmt):
    if fmt != "E4M11_SN":
        raise ValueError(f"Unsupported format: {fmt}")

    encoded = np.asarray(payload, dtype=np.uint8).copy()
    encoded = encoded.view(np.uint16)

    decoded = decode_e4m11(encoded)

    return decoded.reshape(tuple(metadata["shape"])).astype(np.float32)


# ------------------------------------------------------------
# Load original hash references
# ------------------------------------------------------------

with open(HASH_PATH, "r") as f:
    original_hashes = json.load(f)

# ------------------------------------------------------------
# Identify quantizable layers
# ------------------------------------------------------------

layers = []

for name, module in model.named_modules():
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
        layers.append(name)

print("Quantizable layers:", len(layers))
assert len(layers) == 53

# ------------------------------------------------------------
# Compare all 53 payload hashes
# ------------------------------------------------------------

results = []

for layer in layers:
    parameter_name = layer + ".weight"

    weight = dict(model.named_parameters())[parameter_name]

    payload, metadata = encode_weights(weight, "E4M11_SN")

    reconstructed_hash = hashlib.sha256(
        np.ascontiguousarray(payload).tobytes()
    ).hexdigest()

    reference_hash = original_hashes[
        f"E4M11_SN::{layer}"
    ]

    decoded = decode_weights(
        payload, metadata, "E4M11_SN"
    )

    original = weight.detach().cpu().numpy().astype(np.float32)

    mse = float(np.mean((original - decoded) ** 2))
    max_error = float(np.max(np.abs(original - decoded)))

    results.append({
        "layer": layer,
        "payload_bytes": int(len(payload)),
        "reference_hash": reference_hash,
        "reconstructed_hash": reconstructed_hash,
        "hash_match": reconstructed_hash == reference_hash,
        "mse": mse,
        "max_abs_error": max_error,
        "finite": bool(np.all(np.isfinite(decoded)))
    })

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

import pandas as pd

df = pd.DataFrame(results)

matched = int(df["hash_match"].sum())

print("\n" + "=" * 75)
print("E4M11-SN ORIGINAL PAYLOAD VALIDATION")
print("=" * 75)

print("Layers tested:", len(df))
print("Exact hash matches:", matched)
print("Hash mismatches:", len(df) - matched)
print("All reconstructed weights finite:", df["finite"].all())
print("Total payload bytes:", df["payload_bytes"].sum())
print("Mean layer MSE:", df["mse"].mean())
print("Maximum layer MSE:", df["mse"].max())

df.to_csv(OUT / "e4m11_layer_hash_validation.csv", index=False)

report = {
    "format": "E4M11_SN",
    "layers_tested": len(df),
    "hash_matches": matched,
    "hash_mismatches": len(df) - matched,
    "all_hashes_match": bool(df["hash_match"].all()),
    "all_reconstructed_weights_finite": bool(df["finite"].all()),
    "total_payload_bytes": int(df["payload_bytes"].sum()),
    "mean_layer_mse": float(df["mse"].mean()),
    "maximum_layer_mse": float(df["mse"].max())
}

with open(OUT / "e4m11_layer_hash_validation.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nSaved validation artifacts to:", OUT)

if matched == 53:
    print("\nPASS: All 53 E4M11-SN payload hashes match.")
else:
    print("\nFAIL: Codec does not yet reproduce all original payloads.")

Quantizable layers: 53

E4M11-SN ORIGINAL PAYLOAD VALIDATION
Layers tested: 53
Exact hash matches: 53
Hash mismatches: 0
All reconstructed weights finite: True
Total payload bytes: 4635520
Mean layer MSE: 5.7207840624996134e-11
Maximum layer MSE: 4.337183945324341e-10

Saved validation artifacts to: /kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/codec_validation

PASS: All 53 E4M11-SN payload hashes match.


In [22]:

# ================================================================
# JFAMP — ALL 7 DATATYPE CODEC VALIDATION
# 53 layers × 7 formats = 371 reference payload comparisons
# ================================================================

import hashlib
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

# ------------------------------------------------
# 1. PATHS AND CONFIGURATION
# ------------------------------------------------

ROOT = Path(
    "/kaggle/input/notebooks/crahulnaidu/mobile-net-100-3/"
    "JFAMP_MobileNetV2"
)

HASH_PATH = (
    ROOT / "experiments/stage_04_fault_injection/"
    "manifests/stage4b_payload_hashes.json"
)

OUT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/"
    "experiments/revised_jfamp_fault_aware/"
    "codec_validation_all"
)
OUT.mkdir(parents=True, exist_ok=True)

FORMATS = [
    "FP16", "BF16", "E4M11_SN",
    "FP8_E4M3FN", "FP8_E5M2",
    "INT8", "INT4"
]

# ------------------------------------------------
# 2. LOAD REFERENCE HASHES
# ------------------------------------------------

with open(HASH_PATH) as f:
    reference_hashes = json.load(f)

named_parameters = dict(model.named_parameters())

layers = [
    name for name, module in model.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]

assert len(layers) == 53, f"Expected 53 layers, got {len(layers)}"

# ------------------------------------------------
# 3. FLOATING-POINT CODECS
# ------------------------------------------------

def as_numpy(weights):
    if isinstance(weights, torch.Tensor):
        return weights.detach().cpu().numpy().astype(np.float32)
    return np.asarray(weights, dtype=np.float32)


def encode_fp16(x):
    return torch.from_numpy(x.copy()).to(torch.float16).contiguous()


def encode_bf16(x):
    return torch.from_numpy(x.copy()).to(torch.bfloat16).contiguous()


def encode_fp8(x, fmt):
    dtype = (
        torch.float8_e4m3fn
        if fmt == "FP8_E4M3FN"
        else torch.float8_e5m2
    )
    return torch.from_numpy(x.copy()).to(dtype).contiguous()


# E4M11-SN: implementation recovered from user's Stage 2D.2B cell.
EXP_BITS = 4
FRAC_BITS = 11
BIAS = 7
MIN_NORMAL = 2.0 ** (1 - BIAS)
MIN_SUBNORMAL = 2.0 ** (1 - BIAS - FRAC_BITS)

def encode_e4m11(x):
    flat = np.asarray(x, dtype=np.float64).reshape(-1)
    out = np.zeros(flat.shape, dtype=np.uint16)

    for i, value in enumerate(flat):
        sign = int(np.signbit(value))
        sign_bits = sign << 15
        magnitude = abs(value)

        if np.isnan(value):
            out[i] = sign_bits | (15 << FRAC_BITS) | 1
        elif np.isinf(value):
            out[i] = sign_bits | (15 << FRAC_BITS)
        elif magnitude == 0:
            out[i] = sign_bits
        elif magnitude < MIN_NORMAL:
            fraction = int(np.rint(magnitude / MIN_SUBNORMAL))
            if fraction == 0:
                out[i] = sign_bits
            elif fraction >= (1 << FRAC_BITS):
                out[i] = sign_bits | (1 << FRAC_BITS)
            else:
                out[i] = sign_bits | fraction
        else:
            exponent = int(np.floor(np.log2(magnitude)))
            exponent_field = exponent + BIAS
            significand = magnitude / (2.0 ** exponent)
            fraction = int(
                np.rint((significand - 1.0) * (1 << FRAC_BITS))
            )

            if fraction >= (1 << FRAC_BITS):
                fraction = 0
                exponent_field += 1

            if exponent_field >= 15:
                out[i] = sign_bits | (15 << FRAC_BITS)
            else:
                out[i] = (
                    sign_bits
                    | (exponent_field << FRAC_BITS)
                    | fraction
                )

    return out.reshape(np.asarray(x).shape)


# ------------------------------------------------
# 4. INTEGER CODECS — PROBE CONVENTIONS
# ------------------------------------------------

def symmetric_quantize(x, bits):
    qmax = (2 ** (bits - 1)) - 1
    scale = float(np.max(np.abs(x))) / qmax

    if scale == 0:
        scale = 1.0

    q = np.rint(x / scale).clip(-qmax, qmax).astype(np.int8)
    return q, scale


def encode_int8(x):
    q, scale = symmetric_quantize(x, 8)
    payload = q.view(np.uint8).reshape(-1).copy()
    metadata = {"scale": scale, "shape": list(x.shape)}
    return payload, metadata


def encode_int4(x):
    q, scale = symmetric_quantize(x, 4)

    # Two's-complement; first value occupies low nibble.
    signed = q.reshape(-1).astype(np.int16)
    unsigned = (signed & 0x0F).astype(np.uint8)

    if len(unsigned) % 2:
        unsigned = np.append(unsigned, np.uint8(0))

    payload = (
        unsigned[0::2] | (unsigned[1::2] << 4)
    ).astype(np.uint8)

    metadata = {
        "scale": scale,
        "shape": list(x.shape),
        "original_length": int(q.size)
    }

    return payload, metadata


# ------------------------------------------------
# 5. UNIFIED ENCODER
# ------------------------------------------------

def encode_format(weights, fmt):
    x = as_numpy(weights)

    if fmt == "FP16":
        encoded = encode_fp16(x)
        payload = encoded.view(torch.uint8).numpy().reshape(-1).copy()
        metadata = {"shape": list(x.shape)}

    elif fmt == "BF16":
        encoded = encode_bf16(x)
        payload = encoded.view(torch.uint8).numpy().reshape(-1).copy()
        metadata = {"shape": list(x.shape)}

    elif fmt == "E4M11_SN":
        encoded = encode_e4m11(x)
        payload = encoded.view(np.uint8).reshape(-1).copy()
        metadata = {"shape": list(x.shape)}

    elif fmt in ("FP8_E4M3FN", "FP8_E5M2"):
        encoded = encode_fp8(x, fmt)
        payload = encoded.view(torch.uint8).numpy().reshape(-1).copy()
        metadata = {"shape": list(x.shape)}

    elif fmt == "INT8":
        payload, metadata = encode_int8(x)

    elif fmt == "INT4":
        payload, metadata = encode_int4(x)

    else:
        raise ValueError(fmt)

    return payload, metadata


# ------------------------------------------------
# 6. DECODER
# ------------------------------------------------

def decode_format(payload, metadata, fmt):
    raw = np.asarray(payload, dtype=np.uint8).copy()
    shape = tuple(metadata["shape"])

    if fmt == "FP16":
        return torch.from_numpy(raw).view(torch.float16).float().numpy().reshape(shape)

    if fmt == "BF16":
        return torch.from_numpy(raw).view(torch.bfloat16).float().numpy().reshape(shape)

    if fmt == "E4M11_SN":
        words = raw.view(np.uint16)
        result = np.zeros(words.shape, dtype=np.float64)

        for i, word in enumerate(words):
            sign = -1.0 if (int(word) >> 15) & 1 else 1.0
            exponent = (int(word) >> 11) & 0xF
            fraction = int(word) & 0x7FF

            if exponent == 15:
                result[i] = sign * (np.inf if fraction == 0 else np.nan)
            elif exponent == 0:
                result[i] = sign * fraction * MIN_SUBNORMAL
            else:
                result[i] = sign * (
                    1.0 + fraction / 2048.0
                ) * (2.0 ** (exponent - BIAS))

        return result.reshape(shape).astype(np.float32)

    if fmt in ("FP8_E4M3FN", "FP8_E5M2"):
        dtype = (
            torch.float8_e4m3fn
            if fmt == "FP8_E4M3FN"
            else torch.float8_e5m2
        )
        return (
            torch.from_numpy(raw)
            .view(dtype)
            .float()
            .numpy()
            .reshape(shape)
        )

    if fmt == "INT8":
        q = raw.view(np.int8).astype(np.float32)
        return (q * metadata["scale"]).reshape(shape)

    if fmt == "INT4":
        low = raw & 0x0F
        high = (raw >> 4) & 0x0F

        values = np.empty(len(raw) * 2, dtype=np.int8)
        values[0::2] = np.where(low >= 8, low.astype(np.int16) - 16, low)
        values[1::2] = np.where(high >= 8, high.astype(np.int16) - 16, high)

        q = values[:metadata["original_length"]].astype(np.float32)
        return (q * metadata["scale"]).reshape(shape)

    raise ValueError(fmt)


# ------------------------------------------------
# 7. VALIDATE ALL 371 PAYLOADS
# ------------------------------------------------

results = []

print("=" * 90)
print("JFAMP — ALL DATATYPE ORIGINAL PAYLOAD HASH VALIDATION")
print("=" * 90)

for fmt in FORMATS:
    print(f"\nValidating {fmt} ...")

    for layer in layers:
        weight = named_parameters[layer + ".weight"]

        try:
            payload, metadata = encode_format(weight, fmt)

            actual_hash = hashlib.sha256(
                np.ascontiguousarray(payload).tobytes()
            ).hexdigest()

            expected_hash = reference_hashes[f"{fmt}::{layer}"]

            decoded = decode_format(payload, metadata, fmt)

            original = as_numpy(weight)
            finite = bool(np.all(np.isfinite(decoded)))

            mse = float(np.mean((original - decoded) ** 2))
            max_error = float(np.max(np.abs(original - decoded)))

            results.append({
                "format": fmt,
                "layer": layer,
                "payload_bytes": len(payload),
                "hash_match": actual_hash == expected_hash,
                "expected_hash": expected_hash,
                "actual_hash": actual_hash,
                "finite": finite,
                "mse": mse,
                "max_abs_error": max_error,
                "status": "completed"
            })

        except Exception as e:
            results.append({
                "format": fmt,
                "layer": layer,
                "payload_bytes": None,
                "hash_match": False,
                "expected_hash": reference_hashes.get(f"{fmt}::{layer}"),
                "actual_hash": None,
                "finite": False,
                "mse": None,
                "max_abs_error": None,
                "status": f"error: {repr(e)}"
            })

    current = [r for r in results if r["format"] == fmt]
    matches = sum(r["hash_match"] for r in current)
    print(f"  Exact hash matches: {matches}/53")

# ------------------------------------------------
# 8. SAVE RESULTS
# ------------------------------------------------

df = pd.DataFrame(results)

df.to_csv(
    OUT / "all_datatype_layer_hash_validation.csv",
    index=False
)

summary = []

for fmt in FORMATS:
    sub = df[df["format"] == fmt]

    summary.append({
        "format": fmt,
        "layers_tested": len(sub),
        "exact_hash_matches": int(sub["hash_match"].sum()),
        "hash_mismatches": int((~sub["hash_match"]).sum()),
        "all_hashes_match": bool(sub["hash_match"].all()),
        "all_finite": bool(sub["finite"].all()),
        "total_payload_bytes": int(sub["payload_bytes"].fillna(0).sum()),
        "status": (
            "EXACT_MATCH"
            if len(sub) == 53 and sub["hash_match"].all()
            else "NOT_EXACT_OR_FAILED"
        )
    })

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    OUT / "all_datatype_codec_summary.csv",
    index=False
)

report = {
    "experiment": "JFAMP all datatype codec hash validation",
    "layers": len(layers),
    "formats": FORMATS,
    "total_expected_comparisons": len(layers) * len(FORMATS),
    "total_completed_comparisons": len(df),
    "total_exact_hash_matches": int(df["hash_match"].sum()),
    "all_371_hashes_match": bool(
        len(df) == 371 and df["hash_match"].all()
    ),
    "format_summary": summary
}

with open(OUT / "all_datatype_codec_validation.json", "w") as f:
    json.dump(report, f, indent=2)

# ------------------------------------------------
# 9. FINAL OUTPUT
# ------------------------------------------------

print("\n" + "=" * 90)
print("FINAL DATATYPE VALIDATION SUMMARY")
print("=" * 90)

print(summary_df[
    [
        "format",
        "exact_hash_matches",
        "hash_mismatches",
        "all_finite",
        "status"
    ]
].to_string(index=False))

print("\nTotal exact matches:", int(df["hash_match"].sum()), "/ 371")

print("\nArtifacts saved to:")
print(OUT)

print("\nIMPORTANT:")
print("Only formats with 53/53 exact hashes are validated.")
print("A failed hash comparison must not be treated as a validated codec.")


JFAMP — ALL DATATYPE ORIGINAL PAYLOAD HASH VALIDATION

Validating FP16 ...
  Exact hash matches: 53/53

Validating BF16 ...
  Exact hash matches: 53/53

Validating E4M11_SN ...
  Exact hash matches: 53/53

Validating FP8_E4M3FN ...
  Exact hash matches: 53/53

Validating FP8_E5M2 ...
  Exact hash matches: 53/53

Validating INT8 ...
  Exact hash matches: 53/53

Validating INT4 ...
  Exact hash matches: 53/53

FINAL DATATYPE VALIDATION SUMMARY
    format  exact_hash_matches  hash_mismatches  all_finite      status
      FP16                  53                0        True EXACT_MATCH
      BF16                  53                0        True EXACT_MATCH
  E4M11_SN                  53                0        True EXACT_MATCH
FP8_E4M3FN                  53                0        True EXACT_MATCH
  FP8_E5M2                  53                0        True EXACT_MATCH
      INT8                  53                0        True EXACT_MATCH
      INT4                  53                0   

In [25]:

# =====================================================================
# JFAMP — STAGE B: FULL-MODEL LAYER-WISE FAULT SENSITIVITY
#
# Model: MobileNetV2 / CIFAR-100
# Validation samples: 5,000
# Quantizable layers: 53
# Datatypes: 7
# Fault scenarios: 1-bit and 8-bit
# Realizations: 5 per scenario
#
# Planned:
#   371 clean layer-format evaluations
#   3,710 faulted full-model evaluations
#
# Uses previously validated:
#   encode_format(weights, fmt)
#   decode_format(payload, metadata, fmt)
#
# Resume-safe, incremental CSV checkpoints and prediction cache.
# =====================================================================

import os
import json
import copy
import time
import random
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------------------------- CONFIGURATION ----------------------------

MASTER_SEED = 20261114
N_REALIZATIONS = 5

FORMAT_ORDER = [
    "FP16",
    "BF16",
    "E4M11_SN",
    "FP8_E4M3FN",
    "FP8_E5M2",
    "INT8",
    "INT4",
]

FAULT_SCENARIOS = [1, 8]

EXPECTED_LAYERS = 53
EXPECTED_VALIDATION_SAMPLES = 5000
EXPECTED_FP32_ACCURACY = 68.06

ROOT = Path(
    "/kaggle/working/JFAMP_MobileNetV2/experiments/"
    "revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity"
)

ROOT.mkdir(parents=True, exist_ok=True)

PRED_DIR = ROOT / "clean_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_CSV = ROOT / "stageB_clean_layer_format_results.csv"
TRIAL_CSV = ROOT / "stageB_fault_trial_results.csv"
SUMMARY_CSV = ROOT / "stageB_fault_sensitivity_summary.csv"
LAYER_CSV = ROOT / "stageB_layer_summary.csv"
FORMAT_CSV = ROOT / "stageB_format_summary.csv"

BASELINE_JSON = ROOT / "stageB_fp32_baseline.json"
MANIFEST_JSON = ROOT / "stageB_experiment_manifest.json"

# --------------------------- PRE-FLIGHT -------------------------------

required_objects = [
    "model",
    "val_loader",
    "encode_format",
    "decode_format",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing required notebook objects: {missing}. "
        "Please restore the validated Stage 4B codec definitions "
        "and canonical model/validation loader."
    )

if not callable(encode_format):
    raise TypeError("encode_format is not callable.")

if not callable(decode_format):
    raise TypeError("decode_format is not callable.")

if not isinstance(model, nn.Module):
    raise TypeError("model must be a PyTorch model.")

# --------------------------- PAYLOAD HELPER ---------------------------

def to_numpy_payload(payload):
    """
    Convert serialized codec payload to its exact byte representation.

    Important:
    Use view(uint8), not astype(uint8), for non-byte arrays.
    astype would numerically convert values instead of preserving bytes.
    """

    if isinstance(payload, np.ndarray):
        arr = np.ascontiguousarray(payload)

        if arr.dtype == np.uint8:
            return arr.reshape(-1)

        return arr.view(np.uint8).reshape(-1)

    if isinstance(payload, torch.Tensor):
        tensor = payload.detach().cpu().contiguous()

        if tensor.dtype == torch.uint8:
            return tensor.numpy().reshape(-1)

        return tensor.view(torch.uint8).numpy().reshape(-1)

    if isinstance(payload, (bytes, bytearray, memoryview)):
        return np.frombuffer(payload, dtype=np.uint8).copy()

    raise TypeError(
        f"Unsupported serialized payload type: {type(payload)}"
    )


def payload_to_uint8(payload):
    result = np.asarray(
        to_numpy_payload(payload),
        dtype=np.uint8
    ).reshape(-1)

    if result.size == 0:
        raise RuntimeError("Codec returned an empty payload.")

    return np.ascontiguousarray(result)


# --------------------------- REPRODUCIBILITY --------------------------

random.seed(MASTER_SEED)
np.random.seed(MASTER_SEED)
torch.manual_seed(MASTER_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MASTER_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = next(model.parameters()).device

# Make an independent FP32 reference model.
base_model = copy.deepcopy(model).to(device).eval()

non_fp32 = [
    name for name, param in base_model.named_parameters()
    if param.is_floating_point() and param.dtype != torch.float32
]

if non_fp32:
    raise RuntimeError(
        "The reference model contains non-FP32 parameters. "
        f"Examples: {non_fp32[:5]}"
    )

# Discover convolutional and linear layers.
modules = dict(base_model.named_modules())

layer_names = [
    name
    for name, module in modules.items()
    if isinstance(module, (nn.Conv2d, nn.Linear))
    and module.weight is not None
]

if len(layer_names) != EXPECTED_LAYERS:
    raise RuntimeError(
        f"Expected {EXPECTED_LAYERS} quantizable layers, "
        f"found {len(layer_names)}."
    )

try:
    validation_samples = len(val_loader.dataset)
except Exception:
    validation_samples = None

if (
    validation_samples is not None
    and validation_samples != EXPECTED_VALIDATION_SAMPLES
):
    raise RuntimeError(
        f"Expected {EXPECTED_VALIDATION_SAMPLES} validation samples, "
        f"found {validation_samples}."
    )

print("=" * 78)
print("JFAMP STAGE B — PRE-FLIGHT")
print("=" * 78)
print("Device:", device)
print("Quantizable layers:", len(layer_names))
print("Formats:", FORMAT_ORDER)
print("Validation samples:", validation_samples)
print("Fault scenarios:", FAULT_SCENARIOS)
print("Realizations:", N_REALIZATIONS)
print("Output directory:", ROOT)

# --------------------------- HELPER FUNCTIONS -------------------------

def evaluate_full_model(eval_model):
    """Run complete-model evaluation on the canonical validation loader."""

    eval_model.eval()

    predictions = []
    targets = []

    total_loss = 0.0
    total_samples = 0

    with torch.inference_mode():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            logits = eval_model(images)

            loss_sum = F.cross_entropy(
                logits,
                labels,
                reduction="sum"
            )

            batch_size = labels.size(0)

            total_loss += float(loss_sum.item())
            total_samples += batch_size

            predictions.append(
                logits.argmax(dim=1).cpu().numpy()
            )

            targets.append(
                labels.cpu().numpy()
            )

    if total_samples == 0:
        raise RuntimeError("Validation loader returned no samples.")

    predictions = np.concatenate(predictions).astype(np.int64)
    targets = np.concatenate(targets).astype(np.int64)

    if len(predictions) != total_samples:
        raise RuntimeError("Prediction count mismatch.")

    accuracy = (
        100.0
        * float(np.count_nonzero(predictions == targets))
        / total_samples
    )

    return {
        "accuracy_percent": accuracy,
        "loss": total_loss / total_samples,
        "predictions": predictions,
        "targets": targets,
        "correct": predictions == targets,
        "sample_count": total_samples,
    }


def sha256_bytes(array):
    return hashlib.sha256(
        np.ascontiguousarray(array).tobytes()
    ).hexdigest()


def replace_layer_weight(target_model, layer_name, decoded):
    """Replace exactly one layer's weight."""

    target_modules = dict(target_model.named_modules())
    target_module = target_modules[layer_name]

    decoded_tensor = torch.as_tensor(
        decoded,
        dtype=torch.float32,
        device=device
    )

    if tuple(decoded_tensor.shape) != tuple(
        target_module.weight.shape
    ):
        raise RuntimeError(
            f"Shape mismatch in {layer_name}: "
            f"{tuple(decoded_tensor.shape)} vs "
            f"{tuple(target_module.weight.shape)}"
        )

    if not torch.isfinite(decoded_tensor).all():
        raise FloatingPointError(
            f"Non-finite decoded weights in {layer_name}"
        )

    with torch.no_grad():
        target_module.weight.copy_(decoded_tensor)


def append_record(path, record):
    """Append a single record immediately for interruption safety."""

    pd.DataFrame([record]).to_csv(
        path,
        mode="a",
        header=not path.exists(),
        index=False
    )


def prediction_cache_path(layer_idx, format_idx):
    return (
        PRED_DIR
        / f"layer_{layer_idx:02d}_format_{format_idx:02d}.npz"
    )


# --------------------------- FP32 BASELINE ---------------------------

print("\nEvaluating canonical FP32 baseline...")

baseline = evaluate_full_model(base_model)

baseline_record = {
    "accuracy_percent": baseline["accuracy_percent"],
    "loss": baseline["loss"],
    "sample_count": baseline["sample_count"],
    "correct_predictions": int(baseline["correct"].sum()),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

with open(BASELINE_JSON, "w") as f:
    json.dump(baseline_record, f, indent=2)

print(
    f"FP32 accuracy: {baseline_record['accuracy_percent']:.4f}%"
)
print(
    f"FP32 loss: {baseline_record['loss']:.8f}"
)
print(
    f"Correct predictions: "
    f"{baseline_record['correct_predictions']}/"
    f"{baseline_record['sample_count']}"
)

if abs(
    baseline_record["accuracy_percent"] - EXPECTED_FP32_ACCURACY
) > 0.03:
    raise RuntimeError(
        "FP32 baseline does not match the canonical 68.06% result. "
        "Verify checkpoint, data split and preprocessing before continuing."
    )

# --------------------------- RESUME SUPPORT ---------------------------

if CLEAN_CSV.exists():
    existing_clean = pd.read_csv(CLEAN_CSV)

    clean_done = {
        (str(row.layer_name), str(row.format))
        for row in existing_clean.itertuples()
    }
else:
    clean_done = set()

if TRIAL_CSV.exists():
    existing_trials = pd.read_csv(TRIAL_CSV)

    # Retry previous ERROR trials. Only successful and nonfinite
    # trials count as completed for resume purposes.
    completed_trials = {
        (
            str(row.layer_name),
            str(row.format),
            int(row.fault_bits),
            int(row.realization)
        )
        for row in existing_trials.itertuples()
        if str(row.status) in ["OK", "NONFINITE_DECODE"]
    }
else:
    completed_trials = set()

TOTAL_COMBINATIONS = len(layer_names) * len(FORMAT_ORDER)

TOTAL_TRIALS = (
    TOTAL_COMBINATIONS
    * len(FAULT_SCENARIOS)
    * N_REALIZATIONS
)

print("\nResume information:")
print("Existing clean combinations:", len(clean_done))
print("Existing completed trials:", len(completed_trials))
print("Planned clean combinations:", TOTAL_COMBINATIONS)
print("Planned fault trials:", TOTAL_TRIALS)
print("Remaining fault trials:", TOTAL_TRIALS - len(completed_trials))

# --------------------------- MAIN EXPERIMENT --------------------------

start_time = time.time()

new_clean_count = 0
new_trial_count = 0

for layer_idx, layer_name in enumerate(layer_names):

    print("\n" + "=" * 78)
    print(
        f"LAYER {layer_idx + 1}/{len(layer_names)}: {layer_name}"
    )
    print("=" * 78)

    for format_idx, fmt in enumerate(FORMAT_ORDER):

        module = dict(base_model.named_modules())[layer_name]

        # -------------------------------------------------------------
        # 1. Encode the original FP32 layer using validated codec.
        # -------------------------------------------------------------

        source_weights = module.weight.detach().cpu()

        payload, metadata = encode_format(
            source_weights,
            fmt
        )

        clean_payload = payload_to_uint8(payload)

        payload_bytes = int(clean_payload.nbytes)
        payload_bits = payload_bytes * 8

        if payload_bits < max(FAULT_SCENARIOS):
            raise RuntimeError(
                f"Payload too small for requested faults: "
                f"{layer_name}/{fmt}"
            )

        clean_hash = sha256_bytes(clean_payload)

        # -------------------------------------------------------------
        # 2. Load cached clean prediction or evaluate clean model.
        # -------------------------------------------------------------

        pred_path = prediction_cache_path(
            layer_idx,
            format_idx
        )

        if pred_path.exists():

            cached = np.load(pred_path)

            clean_pred = cached["predictions"].astype(np.int64)
            clean_target = cached["targets"].astype(np.int64)
            clean_correct = clean_pred == clean_target
            clean_accuracy = (
                100.0 * float(clean_correct.sum()) / len(clean_target)
            )
            clean_loss = float(cached["loss"])

            if not np.array_equal(
                clean_target,
                baseline["targets"]
            ):
                raise RuntimeError(
                    f"Cached validation labels mismatch: "
                    f"{layer_name}/{fmt}"
                )

        else:

            clean_model = copy.deepcopy(base_model).to(device).eval()

            decoded_clean = decode_format(
                clean_payload.copy(),
                metadata,
                fmt
            )

            replace_layer_weight(
                clean_model,
                layer_name,
                decoded_clean
            )

            clean_result = evaluate_full_model(clean_model)

            clean_pred = clean_result["predictions"]
            clean_target = clean_result["targets"]
            clean_correct = clean_result["correct"]
            clean_accuracy = clean_result["accuracy_percent"]
            clean_loss = clean_result["loss"]

            if not np.array_equal(
                clean_target,
                baseline["targets"]
            ):
                raise RuntimeError(
                    "Validation labels changed between evaluations."
                )

            np.savez_compressed(
                pred_path,
                predictions=clean_pred,
                targets=clean_target,
                loss=np.asarray(clean_loss, dtype=np.float64)
            )

            del clean_model
            del clean_result
            del decoded_clean

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # -------------------------------------------------------------
        # 3. Save clean metrics.
        # -------------------------------------------------------------

        combo_key = (layer_name, fmt)

        if combo_key not in clean_done:

            clean_record = {
                "layer_index": layer_idx,
                "layer_name": layer_name,
                "format": fmt,
                "clean_accuracy_percent": clean_accuracy,
                "clean_loss": clean_loss,
                "accuracy_delta_vs_fp32_pp": (
                    baseline_record["accuracy_percent"]
                    - clean_accuracy
                ),
                "payload_bytes": payload_bytes,
                "payload_bits": payload_bits,
                "clean_payload_sha256": clean_hash,
                "sample_count": len(clean_target),
                "status": "OK",
            }

            append_record(CLEAN_CSV, clean_record)

            clean_done.add(combo_key)
            new_clean_count += 1

        print(
            f"\n{fmt}: clean accuracy={clean_accuracy:.4f}% "
            f"| payload={payload_bytes:,} bytes"
        )

        # -------------------------------------------------------------
        # 4. Inject 1-bit and 8-bit faults.
        # -------------------------------------------------------------

        for fault_bits in FAULT_SCENARIOS:

            for realization in range(
                1,
                N_REALIZATIONS + 1
            ):

                trial_key = (
                    layer_name,
                    fmt,
                    int(fault_bits),
                    int(realization)
                )

                if trial_key in completed_trials:
                    continue

                seed = (
                    MASTER_SEED
                    + layer_idx * 100000
                    + format_idx * 10000
                    + fault_bits * 100
                    + realization
                )

                rng = np.random.default_rng(seed)

                selected_bits = rng.choice(
                    payload_bits,
                    size=fault_bits,
                    replace=False
                )

                corrupted_payload = clean_payload.copy()

                for selected_bit in selected_bits:

                    byte_index = int(selected_bit // 8)
                    bit_index = int(selected_bit % 8)

                    corrupted_payload[byte_index] ^= np.uint8(
                        1 << bit_index
                    )

                verified_flips = int(
                    np.unpackbits(
                        np.bitwise_xor(
                            clean_payload,
                            corrupted_payload
                        )
                    ).sum()
                )

                record = {
                    "layer_index": layer_idx,
                    "layer_name": layer_name,
                    "format": fmt,
                    "fault_bits": int(fault_bits),
                    "realization": int(realization),
                    "seed": int(seed),
                    "payload_bytes": payload_bytes,
                    "payload_bits": payload_bits,
                    "bit_flip_rate": (
                        float(fault_bits) / payload_bits
                    ),
                    "clean_payload_sha256": clean_hash,
                    "corrupted_payload_sha256":
                        sha256_bytes(corrupted_payload),
                    "verified_bit_flips": verified_flips,
                    "selected_bit_positions": json.dumps(
                        [int(x) for x in selected_bits]
                    ),
                    "clean_accuracy_percent": clean_accuracy,
                    "clean_loss": clean_loss,
                    "fault_accuracy_percent": np.nan,
                    "fault_loss": np.nan,
                    "accuracy_degradation_pp": np.nan,
                    "loss_change": np.nan,
                    "changed_predictions": np.nan,
                    "correct_to_wrong": np.nan,
                    "wrong_to_correct": np.nan,
                    "nonfinite_weights": False,
                    "status": "ERROR",
                    "error_message": "",
                }

                try:

                    if verified_flips != fault_bits:
                        raise RuntimeError(
                            f"Expected {fault_bits} bit flips, "
                            f"verified {verified_flips}."
                        )

                    decoded_faulted = decode_format(
                        corrupted_payload,
                        metadata,
                        fmt
                    )

                    decoded_array = np.asarray(
                        decoded_faulted,
                        dtype=np.float32
                    )

                    if not np.isfinite(decoded_array).all():

                        record["nonfinite_weights"] = True
                        record["status"] = "NONFINITE_DECODE"

                    else:

                        # Start from original FP32 model.
                        fault_model = copy.deepcopy(
                            base_model
                        ).to(device).eval()

                        replace_layer_weight(
                            fault_model,
                            layer_name,
                            decoded_array
                        )

                        fault_result = evaluate_full_model(
                            fault_model
                        )

                        fault_pred = fault_result["predictions"]
                        fault_correct = fault_result["correct"]

                        changed = int(
                            np.count_nonzero(
                                fault_pred != clean_pred
                            )
                        )

                        correct_to_wrong = int(
                            np.count_nonzero(
                                clean_correct & ~fault_correct
                            )
                        )

                        wrong_to_correct = int(
                            np.count_nonzero(
                                ~clean_correct & fault_correct
                            )
                        )

                        record.update({
                            "fault_accuracy_percent":
                                fault_result["accuracy_percent"],

                            "fault_loss":
                                fault_result["loss"],

                            "accuracy_degradation_pp":
                                (
                                    clean_accuracy
                                    - fault_result["accuracy_percent"]
                                ),

                            "loss_change":
                                (
                                    fault_result["loss"]
                                    - clean_loss
                                ),

                            "changed_predictions": changed,

                            "correct_to_wrong": correct_to_wrong,

                            "wrong_to_correct": wrong_to_correct,

                            "status": "OK",
                        })

                        del fault_model
                        del fault_result

                    del decoded_faulted
                    del decoded_array

                except FloatingPointError as exc:

                    record["nonfinite_weights"] = True
                    record["status"] = "NONFINITE_DECODE"
                    record["error_message"] = str(exc)

                except Exception as exc:

                    record["status"] = "ERROR"
                    record["error_message"] = (
                        f"{type(exc).__name__}: {str(exc)}"
                    )

                    print(
                        f"ERROR: {layer_name}/{fmt}, "
                        f"{fault_bits}-bit, realization {realization}: "
                        f"{record['error_message']}"
                    )

                # Save every trial immediately.
                append_record(TRIAL_CSV, record)

                if record["status"] in [
                    "OK",
                    "NONFINITE_DECODE"
                ]:
                    completed_trials.add(trial_key)

                new_trial_count += 1

                if record["status"] == "OK":

                    print(
                        f"  {fault_bits}-bit | "
                        f"realization {realization}/{N_REALIZATIONS} | "
                        f"degradation="
                        f"{record['accuracy_degradation_pp']:.4f} pp | "
                        f"changed predictions="
                        f"{record['changed_predictions']}"
                    )

                else:

                    print(
                        f"  {fault_bits}-bit | "
                        f"realization {realization}/{N_REALIZATIONS} | "
                        f"status={record['status']}"
                    )

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

# --------------------------- AGGREGATION -----------------------------

print("\n" + "=" * 78)
print("STAGE B — RESULT AGGREGATION")
print("=" * 78)

if not CLEAN_CSV.exists() or not TRIAL_CSV.exists():
    raise RuntimeError("Expected Stage B result files are missing.")

clean_df = pd.read_csv(CLEAN_CSV)
trials_df = pd.read_csv(TRIAL_CSV)

successful = trials_df[
    trials_df["status"] == "OK"
].copy()

summary_rows = []

for (layer_name, fmt, fault_bits), group in successful.groupby(
    ["layer_name", "format", "fault_bits"],
    sort=False
):

    degradations = (
        group["accuracy_degradation_pp"]
        .dropna()
        .to_numpy()
    )

    if len(degradations) == 0:
        continue

    all_group_trials = trials_df[
        (trials_df["layer_name"] == layer_name)
        & (trials_df["format"] == fmt)
        & (trials_df["fault_bits"] == fault_bits)
    ]

    summary_rows.append({

        "layer_name": layer_name,

        "format": fmt,

        "fault_bits": int(fault_bits),

        "successful_realizations": int(len(degradations)),

        "nonfinite_trials": int(
            (all_group_trials["status"] == "NONFINITE_DECODE").sum()
        ),

        "error_trials": int(
            (all_group_trials["status"] == "ERROR").sum()
        ),

        "expected_degradation_pp": float(
            np.mean(degradations)
        ),

        "median_degradation_pp": float(
            np.median(degradations)
        ),

        "p95_degradation_pp": float(
            np.percentile(degradations, 95)
        ),

        "worst_observed_degradation_pp": float(
            np.max(degradations)
        ),

        "best_observed_degradation_pp": float(
            np.min(degradations)
        ),

        "degradation_std_pp": float(
            np.std(degradations, ddof=1)
        ) if len(degradations) > 1 else 0.0,

        "mean_loss_change": float(
            group["loss_change"].mean()
        ),

        "mean_changed_predictions": float(
            group["changed_predictions"].mean()
        ),

        "mean_correct_to_wrong": float(
            group["correct_to_wrong"].mean()
        ),

        "mean_wrong_to_correct": float(
            group["wrong_to_correct"].mean()
        ),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

# --------------------------- LAYER SUMMARY ----------------------------

layer_rows = []

for layer_name, group in successful.groupby(
    "layer_name",
    sort=False
):

    degradation = (
        group["accuracy_degradation_pp"]
        .dropna()
        .to_numpy()
    )

    if len(degradation) == 0:
        continue

    layer_rows.append({

        "layer_name": layer_name,

        "successful_trials": int(len(degradation)),

        "mean_degradation_pp": float(
            np.mean(degradation)
        ),

        "p95_degradation_pp": float(
            np.percentile(degradation, 95)
        ),

        "worst_observed_degradation_pp": float(
            np.max(degradation)
        ),

        "mean_changed_predictions": float(
            group["changed_predictions"].mean()
        ),
    })

layer_summary_df = pd.DataFrame(layer_rows)
layer_summary_df.to_csv(LAYER_CSV, index=False)

# --------------------------- FORMAT SUMMARY ---------------------------

format_rows = []

for fmt, group in successful.groupby("format", sort=False):

    d = group["accuracy_degradation_pp"].dropna().to_numpy()

    if len(d) == 0:
        continue

    format_rows.append({

        "format": fmt,

        "successful_trials": int(len(d)),

        "mean_degradation_pp": float(np.mean(d)),

        "p95_degradation_pp": float(
            np.percentile(d, 95)
        ),

        "worst_observed_degradation_pp": float(
            np.max(d)
        ),

        "mean_changed_predictions": float(
            group["changed_predictions"].mean()
        ),
    })

format_summary_df = pd.DataFrame(format_rows)
format_summary_df.to_csv(FORMAT_CSV, index=False)

# --------------------------- MANIFEST --------------------------------

manifest = {

    "experiment":
        "JFAMP Stage B — Full-model layer-wise fault sensitivity",

    "timestamp_utc":
        datetime.now(timezone.utc).isoformat(),

    "master_seed":
        MASTER_SEED,

    "model":
        "MobileNetV2 CIFAR-100",

    "canonical_fp32_accuracy_percent":
        baseline_record["accuracy_percent"],

    "canonical_fp32_loss":
        baseline_record["loss"],

    "validation_samples":
        baseline_record["sample_count"],

    "quantizable_layer_count":
        len(layer_names),

    "layer_order":
        layer_names,

    "format_order":
        FORMAT_ORDER,

    "fault_scenarios_bits":
        FAULT_SCENARIOS,

    "realizations_per_scenario":
        N_REALIZATIONS,

    "planned_clean_combinations":
        TOTAL_COMBINATIONS,

    "planned_fault_trials":
        TOTAL_TRIALS,

    "saved_clean_combinations":
        int(len(clean_df)),

    "saved_fault_trials_including_errors":
        int(len(trials_df)),

    "successful_fault_trials":
        int(len(successful)),

    "nonfinite_trials":
        int((trials_df["status"] == "NONFINITE_DECODE").sum()),

    "error_trials":
        int((trials_df["status"] == "ERROR").sum()),

    "seed_formula":
        (
            "MASTER_SEED + layer_index*100000 "
            "+ format_index*10000 + fault_bits*100 + realization"
        ),

    "degradation_convention":
        (
            "clean quantized layer accuracy - faulted full-model accuracy; "
            "positive values indicate degradation"
        ),

    "artifacts": {

        "clean_results": str(CLEAN_CSV),

        "fault_trials": str(TRIAL_CSV),

        "fault_summary": str(SUMMARY_CSV),

        "layer_summary": str(LAYER_CSV),

        "format_summary": str(FORMAT_CSV),

        "clean_prediction_cache": str(PRED_DIR),

        "fp32_baseline": str(BASELINE_JSON),
    },

    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

with open(MANIFEST_JSON, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------- FINAL REPORT ----------------------------

elapsed = time.time() - start_time

clean_complete = len(clean_df) == TOTAL_COMBINATIONS

trial_complete = (
    len(completed_trials) == TOTAL_TRIALS
)

no_errors = (
    int((trials_df["status"] == "ERROR").sum()) == 0
)

experiment_complete = (
    clean_complete
    and trial_complete
    and no_errors
)

print("\n" + "=" * 78)
print("JFAMP STAGE B — FINAL EXECUTION REPORT")
print("=" * 78)

print(
    f"Clean combinations: "
    f"{len(clean_df)}/{TOTAL_COMBINATIONS}"
)

print(
    f"Fault trials recorded: "
    f"{len(trials_df)}/{TOTAL_TRIALS}"
)

print(
    f"Successfully evaluated trials: "
    f"{len(successful)}"
)

print(
    f"Non-finite decode trials: "
    f"{int((trials_df['status'] == 'NONFINITE_DECODE').sum())}"
)

print(
    f"Failed trials: "
    f"{int((trials_df['status'] == 'ERROR').sum())}"
)

print("New clean results:", new_clean_count)
print("New fault trials:", new_trial_count)
print(f"Execution time: {elapsed / 60:.2f} minutes")

if not summary_df.empty:

    print("\nTOP 15 WORST OBSERVED LAYER-FORMAT COMBINATIONS")

    display(
        summary_df.sort_values(
            "worst_observed_degradation_pp",
            ascending=False
        ).head(15).reset_index(drop=True)
    )

    print("\nTOP 15 HIGHEST EXPECTED DEGRADATION")

    display(
        summary_df.sort_values(
            "expected_degradation_pp",
            ascending=False
        ).head(15).reset_index(drop=True)
    )

    print("\nFORMAT-LEVEL SUMMARY")

    display(
        format_summary_df.sort_values(
            "mean_degradation_pp",
            ascending=False
        ).reset_index(drop=True)
    )

print("\nSAVED ARTIFACTS")

for path in [
    CLEAN_CSV,
    TRIAL_CSV,
    SUMMARY_CSV,
    LAYER_CSV,
    FORMAT_CSV,
    BASELINE_JSON,
    MANIFEST_JSON,
]:
    print(f"{path} | exists={path.exists()}")

print("\nSTAGE B STATUS:")

if experiment_complete:
    print("COMPLETE — all planned trials completed without errors.")
else:
    print("PARTIAL / RESUMABLE")
    print(
        "Rerun this same cell after resolving any errors. "
        "Completed successful trials will be skipped."
    )

print("=" * 78)


JFAMP STAGE B — PRE-FLIGHT
Device: cuda:0
Quantizable layers: 53
Formats: ['FP16', 'BF16', 'E4M11_SN', 'FP8_E4M3FN', 'FP8_E5M2', 'INT8', 'INT4']
Validation samples: 5000
Fault scenarios: [1, 8]
Realizations: 5
Output directory: /kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity

Evaluating canonical FP32 baseline...
FP32 accuracy: 68.0600%
FP32 loss: 1.16110593
Correct predictions: 3403/5000

Resume information:
Existing clean combinations: 0
Existing completed trials: 0
Planned clean combinations: 371
Planned fault trials: 3710
Remaining fault trials: 3710

LAYER 1/53: features.0.0

FP16: clean accuracy=68.0600% | payload=1,728 bytes
  1-bit | realization 1/5 | degradation=0.0000 pp | changed predictions=1
  1-bit | realization 2/5 | degradation=0.0000 pp | changed predictions=0
  1-bit | realization 3/5 | degradation=0.0000 pp | changed predictions=0
  1-bit | realization 4/5 | degradation=0.4000 pp | changed predictions=343
  

,layer_name,format,fault_bits,successful_realizations,nonfinite_trials,error_trials,expected_degradation_pp,median_degradation_pp,p95_degradation_pp,worst_observed_degradation_pp,best_observed_degradation_pp,degradation_std_pp,mean_loss_change,mean_changed_predictions,mean_correct_to_wrong,mean_wrong_to_correct
0,features.7.conv.2,FP8_E5M2,1,5,0,0,13.572,0.00,54.064,67.48,-0.02,30.136004,3.456283,1049.2,694.0,15.4
1,features.10.conv.2,BF16,8,5,0,0,26.896,0.00,67.300,67.34,0.00,36.828933,9.872551,1982.4,1351.4,6.6
2,features.5.conv.2,FP8_E5M2,8,5,0,0,40.404,66.90,67.292,67.30,0.22,36.628389,11.774470,3036.4,2044.8,24.6
3,features.7.conv.2,BF16,8,5,0,0,26.744,0.10,67.092,67.28,0.00,36.576619,8.337339,1977.2,1342.8,5.6
4,features.8.conv.2,BF16,1,5,0,0,13.444,0.00,53.792,67.24,-0.02,30.072879,3.881098,993.2,675.8,3.6
5,features.1.conv.1,FP8_E4M3FN,8,5,0,0,27.716,3.70,67.160,67.22,-0.02,35.952050,3.983680,2271.0,1451.0,65.2
6,features.1.conv.1,E4M11_SN,8,5,0,0,40.280,67.06,67.160,67.18,0.02,36.733957,5.363473,2969.6,2020.6,6.6
7,features.6.conv.2,FP8_E5M2,8,5,0,0,53.376,66.40,67.148,67.16,-0.02,29.852080,12.087870,3938.2,2678.4,9.6
8,features.9.conv.2,BF16,8,5,0,0,13.432,0.00,53.728,67.16,0.00,30.034865,3.872870,990.6,674.8,3.2
9,features.4.conv.2,BF16,1,5,0,0,13.480,0.00,53.764,67.14,0.00,29.997063,3.671620,1002.0,678.2,4.2



TOP 15 HIGHEST EXPECTED DEGRADATION


,layer_name,format,fault_bits,successful_realizations,nonfinite_trials,error_trials,expected_degradation_pp,median_degradation_pp,p95_degradation_pp,worst_observed_degradation_pp,best_observed_degradation_pp,degradation_std_pp,mean_loss_change,mean_changed_predictions,mean_correct_to_wrong,mean_wrong_to_correct
0,features.6.conv.2,FP8_E5M2,8,5,0,0,53.376,66.40,67.148,67.16,-0.02,29.852080,12.087870,3938.2,2678.4,9.6
1,features.2.conv.2,FP8_E4M3FN,8,5,0,0,50.516,65.46,66.336,66.40,1.40,28.007429,6.902496,3910.8,2579.6,53.8
2,features.2.conv.2,FP16,8,5,0,0,43.756,66.84,67.060,67.08,-0.10,32.419073,8.052813,3289.8,2212.0,24.2
3,features.1.conv.1,FP8_E5M2,8,5,0,0,40.484,67.10,67.100,67.10,0.02,36.447459,8.715673,3059.0,2045.6,21.4
4,features.4.conv.2,FP8_E5M2,8,5,0,0,40.408,66.78,66.968,66.98,-0.12,36.271020,10.740271,3076.4,2051.0,30.6
5,features.5.conv.2,FP8_E5M2,8,5,0,0,40.404,66.90,67.292,67.30,0.22,36.628389,11.774470,3036.4,2044.8,24.6
6,features.1.conv.1,E4M11_SN,8,5,0,0,40.280,67.06,67.160,67.18,0.02,36.733957,5.363473,2969.6,2020.6,6.6
7,features.4.conv.2,E4M11_SN,8,5,0,0,40.172,51.08,63.800,66.94,0.00,25.716141,4.436265,3193.4,2080.4,71.8
8,features.7.conv.2,FP8_E5M2,8,5,0,0,39.984,66.26,67.020,67.12,-0.04,36.538022,13.355225,2975.8,2015.8,16.6
9,features.6.conv.2,BF16,8,5,0,0,38.472,59.38,66.716,66.86,-0.02,35.250071,9.595950,2845.4,1932.4,8.8



FORMAT-LEVEL SUMMARY


,format,successful_trials,mean_degradation_pp,p95_degradation_pp,worst_observed_degradation_pp,mean_changed_predictions
0,FP8_E5M2,530,7.018906,65.873,67.48,591.164151
1,BF16,530,5.017547,66.250,67.34,414.881132
2,FP16,530,3.991019,45.129,67.08,345.066038
3,FP8_E4M3FN,530,3.685245,33.344,67.22,353.311321
4,E4M11_SN,530,2.703736,14.973,67.18,244.779245
5,INT4,530,0.494792,1.846,32.82,126.249057
6,INT8,530,0.206038,0.420,16.60,64.486792



SAVED ARTIFACTS
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_clean_layer_format_results.csv | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_fault_trial_results.csv | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_fault_sensitivity_summary.csv | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_layer_summary.csv | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_format_summary.csv | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aware/stage_B_layerwise_fault_sensitivity/stageB_fp32_baseline.json | exists=True
/kaggle/working/JFAMP_MobileNetV2/experiments/revised_jfamp_fault_aw